# 04b · 고정 router의 단일·다중과업 학습

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** 동일한 detector와 유형별 경로에서 단일 수위 회귀 arm과 장애물 보조 과업 arm을 비교합니다.
- **입력:** 04A v4 train/validation dataset seal과 shard
- **출력:** 공유 detector, arm별 full-frame/ROI checkpoint와 MODEL_DEPLOYMENT_FROZEN selection seal
- **실제 저장된 실행 상태:** 저장 출력은 single_task 결과 재사용, multitask epoch 2에서 재개한 epoch 3~6 학습 및 v4-fixed-router-multitask 선택 봉인을 기록합니다. multitask epoch 5의 validation RMSE=0.18436134099658713 m가 보이지만, 저장 기록 전체를 처음부터 새로 학습한 실행으로 표현하지 않습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/04b_multitask_training.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/04b_multitask_training.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 04B. YOLO26s·ConvNeXt-Tiny 고정 type router 학습과 선택

이 노트북은 04A의 create-only dataset seal과 train/validation shard만 읽습니다.
v4 bbox로 YOLO26s detector를 한 번 재학습한 뒤 두 실험 arm 사이에서 공유·동결합니다.
B/G/S는 224px full-frame ConvNeXt-Tiny, A/C/E/F/H는 detector가 찾은 mokja를 384px ROI
ConvNeXt-Tiny에 전달합니다. 유효 후보가 없으면 해당 arm의 full-frame checkpoint를 사용합니다.

학습 비교는 단일 수위 회귀 control arm과 수위 회귀·15개 장애물 보조 과업 arm으로 구성하며
각 arm은 full-frame/ROI 회귀 모델을 갖습니다. 네 회귀 모델은 같은 v4 split과 공통
검증 계약을 사용하며, 고정 route로 합성한 배포 후보를 validation 전체 RMSE → 최악 유형
RMSE → 최악 장애물 RMSE 순서로 선택합니다. 익명 test image와 private truth는 이 단계에서
읽지 않습니다.

권장 실행은 먼저 RUN_MODE = "smoke"로 배선과 한두 batch를 확인한 뒤, 새 GPU
런타임에서 RUN_MODE = "full"로 바꾸는 방식입니다. YOLO와 두 ConvNeXt 후보는 완료 epoch마다
GCS에 상태를 commit합니다. 런타임이 종료되면 같은 GPU 종류·RUN_MODE·run ID로 위에서부터
재실행하여 마지막 완료 epoch 다음부터 이어갑니다. 새 런타임에서는 train/validation JPEG
materialization을 다시 수행합니다. full 실행만 MODEL_DEPLOYMENT_FROZEN selection seal을
생성합니다.ㅁ


## 0. Colab dependency bootstrap


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any


def _install_colab_dependencies() -> None:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return
    required = {
        "google.cloud.storage": "google-cloud-storage>=2.19,<4",
        "pyarrow": "pyarrow>=18,<24",
        "pandas": "pandas>=2.2,<3",
        "webdataset": "webdataset>=1.0.2,<2",
        "ultralytics": "ultralytics==8.4.121",
    }
    missing = []
    for module_name, package_spec in required.items():
        try:
            __import__(module_name)
        except ImportError:
            missing.append(package_spec)
    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", *missing]
        )


_install_colab_dependencies()

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import pyarrow as pa
import PIL
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import webdataset as wds
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage
from PIL import Image
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models, transforms

try:
    from google.colab import auth as colab_auth
except ImportError:  # local ADC 환경에서도 정적·소규모 검증 가능
    colab_auth = None


## 1. Immutable configuration


In [ ]:
PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
DATASET_VERSION = "m3ds-v4-ml-fixedtest-v2"
RUN_MODE = "full"  # smoke 확인 후 새 runtime에서 "full"로 명시 변경
BASE_RUN_ID = "m3-v4-yolo26s-fixed-router-obstacle-aux-v3"
RUN_ID = BASE_RUN_ID if RUN_MODE == "full" else f"{BASE_RUN_ID}-smoke"
SELECTION_ID = "m3-selection-v4-router-fixedtest-v4"

CANDIDATE_NAMES = ["single_task", "multitask"]
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 6 if RUN_MODE == "full" else 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TYPE_EMBEDDING_DIM = 32
DROPOUT = 0.15
OBSTACLE_LOSS_WEIGHT = 0.10
MAX_POS_WEIGHT = 20.0
AUGMENT = False
SEED = 42
MAX_TRAIN_BATCHES = None if RUN_MODE == "full" else 2
MAX_VAL_BATCHES = None  # smoke는 유형별 소규모 validation subset 전체를 평가
MIN_TYPE_OBSTACLE_SLICE_N = 30
ROUTED_CROP_PADDING = 0.15
ROI_TRAIN_JITTER = 0.04
YOLO_IMGSZ, YOLO_BATCH = 960, 4
YOLO_CONFIDENCE, YOLO_IOU_THRESHOLD, YOLO_MAX_DET = 0.05, 0.70, 10
FROZEN_DETECTOR_OBJECT = "dcc/mission3/runs/m3-yolo26s-roi-router-full-v2/detector/best.pt"
FROZEN_DETECTOR_ALLOWED_ARTIFACT_TYPES = {"m3_yolo26_best_checkpoint", "m3_yolo26_checkpoint"}

TYPE_ORDER = list("ABCEFGHS")
TYPE_TO_INDEX = {name: index for index, name in enumerate(TYPE_ORDER)}
TYPE_RANGES = {
    "A": (0.0, 15.0),
    "B": (0.0, 9.0),
    "C": (0.0, 15.0),
    "E": (0.0, 10.0),
    "F": (0.0, 4.0),
    "G": (0.0, 4.0),
    "H": (0.0, 20.0),
    "S": (0.0, 20.0),
}
OBSTACLE_ORDER = [
    "거센물살",
    "거울반사",
    "기타",
    "달빛반사",
    "반투명",
    "부유물",
    "부착물",
    "사람",
    "수풀",
    "안개",
    "탁류",
    "표면손상",
    "표면오염",
    "야간",
    "와류",
]

LOCAL_ROOT = Path("/content/dcc_mission3_v4_training")
LOCAL_SHARDS = LOCAL_ROOT / "shards" / DATASET_VERSION
LOCAL_RUN = LOCAL_ROOT / "runs" / RUN_ID
LOCAL_META = LOCAL_ROOT / "meta"
LOCAL_IMAGES = LOCAL_ROOT / "images"
LOCAL_ROUTING = LOCAL_ROOT / "routing"
for _path in [LOCAL_SHARDS, LOCAL_RUN, LOCAL_META, LOCAL_IMAGES, LOCAL_ROUTING]:
    _path.mkdir(parents=True, exist_ok=True)

DATASET_SEAL_OBJECT = f"dcc/mission3/datasets/{DATASET_VERSION}/dataset_seal.json"
RUN_PREFIX = f"dcc/mission3/v4/runs/{RUN_ID}"
SELECTION_OBJECT = (
    f"dcc/mission3/v4/selection/{SELECTION_ID}/model_selection_seal.json"
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("04B smoke와 full 실행에는 GPU runtime이 필요합니다")

def enforce_deterministic_runtime() -> None:
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")


enforce_deterministic_runtime()

RUNTIME_CONTRACT = {
    "schema_version": "dcc-m3-v4-exact-resume-runtime-v1",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pa.__version__,
    "pillow": PIL.__version__,
    "torch": str(torch.__version__),
    "torchvision": torchvision.__version__,
    "cuda": str(torch.version.cuda),
    "cudnn": str(torch.backends.cudnn.version()),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
    "cuda_device_count": torch.cuda.device_count(),
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    "deterministic_warn_only": torch.is_deterministic_algorithms_warn_only_enabled(),
    "float32_matmul_precision": torch.get_float32_matmul_precision(),
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "cuda_matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
    "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)


## 2. Hash, GCS, and create-only helpers


In [ ]:
def sha256_file(path: str | Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def canonical_sha256(value: Any) -> str:
    raw = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


RUNTIME_CONTRACT_SHA256 = canonical_sha256(RUNTIME_CONTRACT)


def canonical_target_sha256(frame: pd.DataFrame) -> str:
    required = {"stem", "type", "target_m"}
    if required - set(frame.columns):
        raise AssertionError(f"target hash 열 누락: {sorted(required - set(frame.columns))}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(row.stem),
            "type": str(row.type),
            "target_m_float32": float(np.float32(row.target_m)),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)


def canonical_obstruction_sha256(frame: pd.DataFrame) -> str:
    required = {"stem", "obstruction_targets", "obstruction_mask"}
    if required - set(frame.columns):
        raise AssertionError(
            f"obstruction hash 열 누락: {sorted(required - set(frame.columns))}"
        )
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(row.stem),
            "obstruction_targets": [int(x) for x in row.obstruction_targets],
            "obstruction_mask": [int(x) for x in row.obstruction_mask],
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)


def parse_gs_uri(uri: str) -> tuple[str, str]:
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    return tuple(str(uri)[5:].split("/", 1))  # type: ignore[return-value]


def download_object(
    bucket: storage.Bucket,
    object_name: str,
    local_path: str | Path,
    generation: int | None = None,
) -> Path:
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs: dict[str, Any] = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path


def _artifact_metadata(
    file_sha: str,
    artifact_type: str,
    object_metadata: dict[str, Any] | None = None,
) -> dict[str, str]:
    metadata = {"sha256": file_sha, "artifact_type": artifact_type}
    metadata.update({str(key): str(value) for key, value in (object_metadata or {}).items()})
    return metadata


def _metadata_contains(actual: dict[str, str], expected: dict[str, str]) -> bool:
    return all(str(actual.get(key)) == str(value) for key, value in expected.items())


def upload_create_only(
    bucket: storage.Bucket,
    local_path: str | Path,
    object_name: str,
    artifact_type: str,
    object_metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    expected_metadata = _artifact_metadata(file_sha, artifact_type, object_metadata)
    blob = bucket.blob(object_name)
    blob.metadata = expected_metadata
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        if (
            not _metadata_contains(blob.metadata or {}, expected_metadata)
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(f"create-only 충돌: gs://{bucket.name}/{object_name}")
    blob.reload()
    if (
        int(blob.size or -1) != local_path.stat().st_size
        or not _metadata_contains(blob.metadata or {}, expected_metadata)
    ):
        raise RuntimeError(f"업로드 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }


def upload_json_create_only(
    bucket: storage.Bucket,
    payload: dict[str, Any],
    object_name: str,
    artifact_type: str,
    object_metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    local_path = LOCAL_META / f"{artifact_type}-{Path(object_name).name}"
    local_path.write_text(
        json.dumps(
            payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False
        )
        + "\n",
        encoding="utf-8",
    )
    return upload_create_only(
        bucket, local_path, object_name, artifact_type, object_metadata
    )


def remote_descriptor(
    bucket: storage.Bucket, object_name: str, artifact_type: str
) -> dict[str, Any]:
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    if metadata.get("artifact_type") != artifact_type or not metadata.get("sha256"):
        raise AssertionError(f"remote descriptor metadata 오류: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": metadata["sha256"],
        "artifact_type": artifact_type,
    }


def try_remote_descriptor(
    bucket: storage.Bucket,
    object_name: str,
    artifact_type: str,
    required_metadata: dict[str, Any] | None = None,
) -> dict[str, Any] | None:
    blob = bucket.blob(object_name)
    try:
        blob.reload()
    except NotFound:
        return None
    metadata = blob.metadata or {}
    expected = {
        "artifact_type": artifact_type,
        **{str(key): str(value) for key, value in (required_metadata or {}).items()},
    }
    if not metadata.get("sha256") or not _metadata_contains(metadata, expected):
        raise RuntimeError(
            f"원격 artifact 계약 충돌: gs://{bucket.name}/{object_name}"
        )
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": metadata["sha256"],
        "artifact_type": artifact_type,
    }


def validate_pinned_descriptor(
    bucket: storage.Bucket,
    descriptor: dict[str, Any],
    required_metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    if bucket_name != bucket.name:
        raise AssertionError(f"descriptor bucket 불일치: {descriptor['uri']}")
    blob = bucket.blob(object_name, generation=int(descriptor["generation"]))
    blob.reload()
    expected = {
        "sha256": str(descriptor["sha256"]),
        "artifact_type": str(descriptor["artifact_type"]),
        **{str(key): str(value) for key, value in (required_metadata or {}).items()},
    }
    if (
        int(blob.size or -1) != int(descriptor["bytes"])
        or not _metadata_contains(blob.metadata or {}, expected)
    ):
        raise AssertionError(f"pinned descriptor 검증 실패: {descriptor['uri']}")
    return descriptor


def download_verified_descriptor(
    bucket: storage.Bucket,
    descriptor: dict[str, Any],
    local_path: str | Path,
    required_metadata: dict[str, Any] | None = None,
) -> Path:
    validate_pinned_descriptor(bucket, descriptor, required_metadata)
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    if bucket_name != bucket.name:
        raise AssertionError("descriptor bucket 불일치")
    local_path = download_object(
        bucket, object_name, local_path, int(descriptor["generation"])
    )
    if (
        local_path.stat().st_size != int(descriptor["bytes"])
        or sha256_file(local_path) != descriptor["sha256"]
    ):
        raise AssertionError(f"download descriptor byte/SHA 불일치: {descriptor['uri']}")
    return local_path


def load_remote_json(
    bucket: storage.Bucket,
    descriptor: dict[str, Any],
    local_path: str | Path,
    required_metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    path = download_verified_descriptor(
        bucket, descriptor, local_path, required_metadata
    )
    return json.loads(path.read_text(encoding="utf-8"))


def load_remote_torch(
    bucket: storage.Bucket,
    descriptor: dict[str, Any],
    local_path: str | Path,
    required_metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    path = download_verified_descriptor(
        bucket, descriptor, local_path, required_metadata
    )
    return torch.load(path, map_location="cpu", weights_only=False)


if colab_auth is not None:
    colab_auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()

DATASET_SEAL_DESCRIPTOR = remote_descriptor(
    BUCKET, DATASET_SEAL_OBJECT, "m3_v4_dataset_seal"
)
dataset_seal_path = download_object(
    BUCKET,
    DATASET_SEAL_OBJECT,
    LOCAL_META / "dataset_seal.json",
    generation=DATASET_SEAL_DESCRIPTOR["generation"],
)
if sha256_file(dataset_seal_path) != DATASET_SEAL_DESCRIPTOR["sha256"]:
    raise AssertionError("dataset seal SHA 불일치")
dataset_seal = json.loads(dataset_seal_path.read_text(encoding="utf-8"))
if dataset_seal.get("status") != "DATASET_COMPLETE":
    raise AssertionError("dataset seal 상태 오류")
if dataset_seal.get("schema_version") != "dcc-m3-dataset-seal-v2":
    raise AssertionError("v4 dataset seal schema 오류")
if dataset_seal.get("dataset_version") != DATASET_VERSION:
    raise AssertionError("dataset version 불일치")
if dataset_seal.get("data_profile") != "full":
    raise AssertionError("v4 학습은 full profile만 허용합니다")
if not all(dataset_seal.get("assertions", {}).values()):
    raise AssertionError("upstream dataset seal assertion 실패")
obstruction_schema = dataset_seal.get("obstruction_schema", {})
if obstruction_schema.get("labels") != OBSTACLE_ORDER:
    raise AssertionError("obstruction vocabulary/order 불일치")
if obstruction_schema.get("encoding") != "multi_hot_15_with_known_mask":
    raise AssertionError("obstruction encoding 불일치")
if obstruction_schema.get("vocabulary_sha256") != canonical_sha256(OBSTACLE_ORDER):
    raise AssertionError("obstruction vocabulary SHA 불일치")
if "validation_obstruction_sha256" not in dataset_seal:
    raise AssertionError("validation obstruction hash 누락")
TEST_CONTRACT = dataset_seal.get("test_contract")
if not isinstance(TEST_CONTRACT, dict):
    raise AssertionError("fixed-test contract 누락")
required_test_contract = {
    "inference_image_seal_sha256", "test_sample_id_sha256", "row_count"
}
if required_test_contract - set(TEST_CONTRACT):
    raise AssertionError("fixed-test contract field 누락")
for hash_field in ["inference_image_seal_sha256", "test_sample_id_sha256"]:
    value = str(TEST_CONTRACT[hash_field])
    if len(value) != 64 or any(ch not in "0123456789abcdef" for ch in value):
        raise AssertionError(f"fixed-test {hash_field} 형식 오류")
if int(TEST_CONTRACT["row_count"]) <= 0:
    raise AssertionError("fixed-test row_count 오류")

EXPECTED_TRAIN_N = int(dataset_seal["counts"]["train"])
EXPECTED_VAL_N = int(dataset_seal["counts"]["val"])
if EXPECTED_TRAIN_N <= 0 or EXPECTED_VAL_N <= 0:
    raise AssertionError("train/validation count 오류")


## 3. Shard descriptors are streamed after the sample validator is defined.


In [ ]:
# No stage downloads every tar before extraction. Each pinned shard is downloaded,
# checked, extracted, and removed before the next shard starts.



import ultralytics
if ultralytics.__version__ != "8.4.121":
    raise RuntimeError(f"ultralytics version mismatch: {ultralytics.__version__}")
from ultralytics import YOLO
from torch.utils.data import DataLoader, Dataset
COMPONENT_NAMES = ["full_frame", "roi"]
ROI_TYPES = frozenset("ACEFH")
FULL_FRAME_TYPES = frozenset("BGS")
ROUTE_BY_TYPE = {**{x: "yolo_roi" for x in ROI_TYPES}, **{x: "full_frame" for x in FULL_FRAME_TYPES}}
ROI_IMAGE_SIZE = 384
ROI_BATCH_SIZE = 16
BASE_RUN_ID = "m3-v4-yolo26s-fixed-router-obstacle-aux-v3"
RUN_ID = BASE_RUN_ID if RUN_MODE == "full" else f"{BASE_RUN_ID}-smoke"
SELECTION_ID = "m3-selection-v4-router-fixedtest-v4"
LOCAL_RUN = LOCAL_ROOT / "runs" / RUN_ID
LOCAL_IMAGES = LOCAL_ROOT / "images"
LOCAL_ROUTING = LOCAL_ROOT / "routing"
for _extra in [LOCAL_RUN, LOCAL_IMAGES, LOCAL_ROUTING]:
    _extra.mkdir(parents=True, exist_ok=True)
RUN_PREFIX = f"dcc/mission3/v4/runs/{RUN_ID}"
SELECTION_OBJECT = f"dcc/mission3/v4/selection/{SELECTION_ID}/model_selection_seal.json"


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 4. v4 detector retraining, immutable routing contract, and route cache


In [ ]:
import tarfile
from collections import defaultdict

DETECTOR_MODE = "retrain_v4"  # "retrain_v4" (full default) or "reuse_v3"
DETECTOR_BASE_WEIGHTS = "yolo26s.pt"
DETECTOR_EPOCHS = 4 if RUN_MODE == "full" else 1
DETECTOR_TRAIN_CAP = 60_000 if RUN_MODE == "full" else 512
DETECTOR_RUN_ID = f"{RUN_ID}-detector"
if RUN_MODE == "full" and DETECTOR_MODE != "retrain_v4":
    raise AssertionError("full 기본 실행은 v4 bbox detector 재학습이어야 합니다")
if set(ROUTE_BY_TYPE) != set(TYPE_ORDER):
    raise AssertionError("route type 집합 오류")


def obstacle_arrays(metadata):
    target = np.asarray(metadata["obstruction_targets"], dtype=np.float32)
    known = np.asarray(metadata["obstruction_mask"], dtype=np.float32)
    if target.shape != (len(OBSTACLE_ORDER),) or known.shape != target.shape:
        raise AssertionError("obstruction vector shape 오류")
    if not np.isin(target, [0.0, 1.0]).all() or not np.isin(known, [0.0, 1.0]).all():
        raise AssertionError("obstruction vector value 오류")
    return target, known


STREAMING_DISK_SAFETY_BYTES = 12 * 1024**3
ALL_TRAIN_VAL_DESCRIPTORS = [
    descriptor
    for split_name in ["train", "val"]
    for descriptor in dataset_seal["shards"][split_name]
]
TOTAL_DESCRIPTOR_BYTES = sum(
    int(descriptor["bytes"]) for descriptor in ALL_TRAIN_VAL_DESCRIPTORS
)
MAX_SHARD_BYTES = max(
    int(descriptor["bytes"]) for descriptor in ALL_TRAIN_VAL_DESCRIPTORS
)
INITIAL_FREE_BYTES = shutil.disk_usage("/content").free
if INITIAL_FREE_BYTES < (
    TOTAL_DESCRIPTOR_BYTES + MAX_SHARD_BYTES + STREAMING_DISK_SAFETY_BYTES
):
    raise RuntimeError(
        "train/validation JPEG materialization을 위한 local disk가 부족합니다: "
        f"free={INITIAL_FREE_BYTES:,}, estimated_payload={TOTAL_DESCRIPTOR_BYTES:,}, "
        f"largest_shard={MAX_SHARD_BYTES:,}, safety={STREAMING_DISK_SAFETY_BYTES:,}"
    )


def materialize_split_streaming(split_name: str) -> pd.DataFrame:
    descriptors = list(dataset_seal["shards"][split_name])
    if not descriptors:
        raise AssertionError(f"{split_name} shard descriptor 누락")
    root = LOCAL_IMAGES / split_name
    root.mkdir(parents=True, exist_ok=True)
    records = []
    seen_stems = set()
    descriptor_sample_total = 0
    for shard_i, descriptor in enumerate(descriptors, start=1):
        bucket_name, object_name = parse_gs_uri(descriptor["uri"])
        if bucket_name != BUCKET_NAME:
            raise AssertionError("shard bucket 불일치")
        if descriptor.get("artifact_type") != "m3_v4_training_image_shard":
            raise AssertionError("shard artifact_type 불일치")
        if not object_name.startswith(
            f"dcc/mission3/datasets/{DATASET_VERSION}/"
        ):
            raise AssertionError("shard object prefix 불일치")
        expected_samples = int(descriptor["samples"])
        if expected_samples <= 0:
            raise AssertionError("shard sample count 오류")
        local_shard = (
            LOCAL_SHARDS
            / split_name
            / descriptor["sha256"][:16]
            / Path(object_name).name
        )
        if local_shard.exists() and (
            local_shard.stat().st_size != int(descriptor["bytes"])
            or sha256_file(local_shard) != descriptor["sha256"]
        ):
            raise RuntimeError(f"기존 local shard 충돌: {local_shard}")
        if not local_shard.exists():
            download_object(
                BUCKET,
                object_name,
                local_shard,
                generation=int(descriptor["generation"]),
            )
        if (
            local_shard.stat().st_size != int(descriptor["bytes"])
            or sha256_file(local_shard) != descriptor["sha256"]
        ):
            raise AssertionError("downloaded shard byte/SHA 검증 실패")

        shard_records = 0
        with tarfile.open(local_shard, "r:") as handle:
            regular_members = [
                member for member in handle.getmembers() if member.isfile()
            ]
            member_names = [member.name for member in regular_members]
            if len(member_names) != len(set(member_names)):
                raise AssertionError("tar member name 중복")
            if any("/" in name or "\\" in name for name in member_names):
                raise AssertionError("tar nested path 금지")
            member_by_name = {member.name: member for member in regular_members}
            json_stems = {
                Path(name).stem for name in member_names if name.endswith(".json")
            }
            image_stems = {
                Path(name).stem for name in member_names if name.endswith(".jpg")
            }
            if json_stems != image_stems:
                raise AssertionError("tar jpg/json pair 불일치")
            if len(json_stems) != expected_samples:
                raise AssertionError("descriptor/tar sample count 불일치")
            for stem in sorted(json_stems):
                if stem in seen_stems:
                    raise AssertionError(f"split 내 stem 중복: {stem}")
                raw_metadata = handle.extractfile(
                    member_by_name[f"{stem}.json"]
                ).read()
                metadata = json.loads(raw_metadata.decode("utf-8"))
                image_bytes = handle.extractfile(
                    member_by_name[f"{stem}.jpg"]
                ).read()
                if metadata["stem"] != stem or metadata["split"] != split_name:
                    raise AssertionError("tar key/split 불일치")
                if hashlib.sha256(image_bytes).hexdigest() != metadata["image_sha256"]:
                    raise AssertionError("image SHA 불일치")
                image_path = root / f"{stem}.jpg"
                if not image_path.exists():
                    with open(image_path, "xb") as stream:
                        stream.write(image_bytes)
                elif (
                    image_path.stat().st_size != len(image_bytes)
                    or sha256_file(image_path) != metadata["image_sha256"]
                ):
                    raise RuntimeError("기존 image 충돌")
                obstacle_target, obstacle_known = obstacle_arrays(metadata)
                records.append({
                    "stem": stem,
                    "type": str(metadata["type"]),
                    "split": split_name,
                    "group_id": str(metadata["group_id"]),
                    "image_sha256": str(metadata["image_sha256"]),
                    "image_path": str(image_path),
                    "target_m": float(metadata["target_m"]),
                    "detector_trainable": bool(metadata["detector_trainable"]),
                    "boxes": list(metadata["mokja_boxes_xyxy_norm"]),
                    "obstruction_targets": obstacle_target.astype(np.int8).tolist(),
                    "obstruction_mask": obstacle_known.astype(np.int8).tolist(),
                })
                seen_stems.add(stem)
                shard_records += 1
        if shard_records != expected_samples:
            raise AssertionError("shard materialized sample count 불일치")
        descriptor_sample_total += expected_samples
        # Fully verified tar is removed before the next descriptor is downloaded.
        local_shard.unlink(missing_ok=True)
        if local_shard.exists():
            raise RuntimeError("verified local shard cleanup 실패")
        if shard_i % 10 == 0 or shard_i == len(descriptors):
            print("stream materialize", split_name, shard_i, "/", len(descriptors))

    expected = EXPECTED_TRAIN_N if split_name == "train" else EXPECTED_VAL_N
    if descriptor_sample_total != expected:
        raise AssertionError(f"{split_name} descriptor sample total 불일치")
    frame = pd.DataFrame(records).sort_values(
        "stem", kind="mergesort"
    ).reset_index(drop=True)
    if len(frame) != expected or not frame["stem"].is_unique:
        raise AssertionError("materialized split count/uniqueness 오류")
    return frame


TRAIN_META = materialize_split_streaming("train")
VAL_META = materialize_split_streaming("val")
if canonical_sha256(sorted(VAL_META["stem"])) != dataset_seal["validation_stem_sha256"]:
    raise AssertionError("validation stem SHA 불일치")
if canonical_target_sha256(VAL_META) != dataset_seal["validation_target_sha256"]:
    raise AssertionError("validation target SHA 불일치")
if canonical_obstruction_sha256(VAL_META) != dataset_seal["validation_obstruction_sha256"]:
    raise AssertionError("validation obstruction SHA 불일치")
if pd.concat([TRAIN_META, VAL_META]).groupby("group_id")["split"].nunique().max() != 1:
    raise AssertionError("group split leakage")


def yolo_line(box):
    x1, y1, x2, y2 = map(float, box)
    if not all(math.isfinite(x) for x in [x1, y1, x2, y2]):
        return None
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(1, x2), min(1, y2)
    if x2 <= x1 or y2 <= y1:
        return None
    return "0 " + " ".join(f"{x:.8f}" for x in [
        (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
    ])


def select_detector_train(frame: pd.DataFrame) -> pd.DataFrame:
    eligible = frame.loc[
        frame["detector_trainable"]
        & frame["boxes"].map(lambda boxes: any(yolo_line(box) is not None for box in boxes))
    ].copy()
    eligible["target_decile"] = (
        eligible.groupby("type", observed=True)["target_m"]
        .transform(lambda x: pd.qcut(x.rank(method="first"), 10, labels=False))
        .astype(int)
    )
    eligible["_priority"] = [
        hashlib.sha256(f"{SEED}|{t}|{d}|{s}".encode()).hexdigest()
        for t, d, s in zip(eligible["type"], eligible["target_decile"], eligible["stem"])
    ]
    if len(eligible) <= DETECTOR_TRAIN_CAP:
        return eligible
    sizes = eligible.groupby(["type", "target_decile"], observed=True).size()
    raw_quota = sizes / sizes.sum() * DETECTOR_TRAIN_CAP
    quota = np.floor(raw_quota).astype(int)
    remainder = DETECTOR_TRAIN_CAP - int(quota.sum())
    for key in (raw_quota - quota).sort_values(ascending=False).index[:remainder]:
        quota.loc[key] += 1
    selected = pd.concat([
        part.sort_values("_priority", kind="stable").head(int(quota.loc[key]))
        for key, part in eligible.groupby(["type", "target_decile"], observed=True)
    ], ignore_index=True)
    if len(selected) != DETECTOR_TRAIN_CAP:
        raise AssertionError("detector cap 오류")
    return selected


DETECTOR_TRAIN_META = select_detector_train(TRAIN_META)
if RUN_MODE == "full" and len(DETECTOR_TRAIN_META) != DETECTOR_TRAIN_CAP:
    raise AssertionError("detector train cap 60,000 hard gate 실패")
DETECTOR_VAL_META = VAL_META.loc[
    VAL_META["detector_trainable"]
    & VAL_META["boxes"].map(lambda boxes: any(yolo_line(box) is not None for box in boxes))
].copy()
if RUN_MODE != "full":
    DETECTOR_VAL_META = (
        DETECTOR_VAL_META.groupby("type", sort=False, group_keys=False)
        .head(16).reset_index(drop=True)
    )
YOLO_ROOT = LOCAL_ROOT / "yolo_v4"
for split_name, frame in [("train", DETECTOR_TRAIN_META), ("val", DETECTOR_VAL_META)]:
    image_dir, label_dir = YOLO_ROOT / "images" / split_name, YOLO_ROOT / "labels" / split_name
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    for row in frame.itertuples(index=False):
        lines = [line for line in (yolo_line(box) for box in row.boxes) if line is not None]
        if not lines:
            continue
        source, target = Path(row.image_path), image_dir / f"{row.stem}.jpg"
        if not target.exists():
            os.link(source, target)
        (label_dir / f"{row.stem}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
for split_name, expected_rows in [
    ("train", len(DETECTOR_TRAIN_META)), ("val", len(DETECTOR_VAL_META))
]:
    image_stems = {path.stem for path in (YOLO_ROOT/"images"/split_name).glob("*.jpg")}
    label_stems = {path.stem for path in (YOLO_ROOT/"labels"/split_name).glob("*.txt")}
    if image_stems != label_stems or len(image_stems) != expected_rows:
        raise AssertionError(f"YOLO {split_name} image/label/count hard gate 실패")
if not len(DETECTOR_VAL_META):
    raise AssertionError("YOLO validation rows empty")

YOLO_DATA_YAML = YOLO_ROOT / "data.yaml"
YOLO_DATA_YAML.write_text(
    f"path: {YOLO_ROOT}\ntrain: images/train\nval: images/val\nnames:\n  0: mokja\n",
    encoding="utf-8",
)
DETECTOR_CONFIG = {
    "schema_version": "dcc-m3-v4-yolo26s-detector-config-v1",
    "mode": DETECTOR_MODE, "base_weights": DETECTOR_BASE_WEIGHTS,
    "ultralytics_version": "8.4.121", "class_names": ["mokja"],
    "epochs": DETECTOR_EPOCHS, "train_cap": DETECTOR_TRAIN_CAP,
    "train_rows": len(DETECTOR_TRAIN_META), "val_rows": len(DETECTOR_VAL_META),
    "train_selection": "proportional_type_x_target_decile_sha256_seeded",
    "selection": "proportional_type_x_target_decile_sha256_seeded",
    "imgsz": YOLO_IMGSZ, "batch": YOLO_BATCH,
    "conf": YOLO_CONFIDENCE, "iou": YOLO_IOU_THRESHOLD,
    "max_det": YOLO_MAX_DET, "seed": SEED,
    "amp": True, "cache": False, "fraction": 1.0,
    "augmentation_off": ["mosaic","mixup","cutmix","copy_paste","multi_scale","close_mosaic","fliplr","flipud","hsv_h","hsv_s","hsv_v","degrees","translate","scale","shear","perspective"],
    "dataset_version": DATASET_VERSION,
    "resume_contract": "gcs_content_addressed_epoch_manifest_v1",
}
DETECTOR_CONFIG_SHA256 = canonical_sha256(DETECTOR_CONFIG)
DETECTOR_RESUME_PREFIX = f"{RUN_PREFIX}/detector/resume/{DETECTOR_CONFIG_SHA256}"
DETECTOR_COMPLETION_OBJECT = f"{DETECTOR_RESUME_PREFIX}/completion.json"
DETECTOR_LAST_TYPE = "m3_v4_yolo26s_resume_checkpoint"
DETECTOR_BEST_TYPE = "m3_v4_yolo26s_best_checkpoint"
DETECTOR_EPOCH_TYPE = "m3_v4_yolo26s_epoch_manifest"
DETECTOR_COMPLETION_TYPE = "m3_v4_yolo26s_completion_manifest"


def load_detector_json(
    object_name: str, artifact_type: str,
) -> tuple[dict[str, Any], dict[str, Any]] | None:
    blob = BUCKET.blob(object_name)
    if not blob.exists(client=STORAGE_CLIENT):
        return None
    descriptor = remote_descriptor(BUCKET, object_name, artifact_type)
    local = LOCAL_META / "detector-resume" / Path(object_name).name
    download_object(BUCKET, object_name, local, descriptor["generation"])
    if local.stat().st_size != descriptor["bytes"] or sha256_file(local) != descriptor["sha256"]:
        raise AssertionError(f"detector JSON SHA/size 오류: {object_name}")
    payload = json.loads(local.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise AssertionError(f"detector JSON payload 오류: {object_name}")
    return payload, descriptor


def validate_detector_pt_descriptor(descriptor: dict[str, Any], role: str) -> str:
    artifact_type = {"last": DETECTOR_LAST_TYPE, "best": DETECTOR_BEST_TYPE}[role]
    required = {"uri", "generation", "bytes", "sha256", "artifact_type"}
    if not isinstance(descriptor, dict) or required - set(descriptor):
        raise AssertionError(f"detector {role} descriptor 오류")
    file_sha = str(descriptor["sha256"])
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    expected = f"{DETECTOR_RESUME_PREFIX}/{role}/{file_sha}.pt"
    if (
        descriptor["artifact_type"] != artifact_type
        or len(file_sha) != 64
        or any(ch not in "0123456789abcdef" for ch in file_sha)
        or bucket_name != BUCKET_NAME
        or object_name != expected
    ):
        raise AssertionError(f"detector {role} descriptor contract 오류")
    actual = remote_descriptor(BUCKET, object_name, artifact_type)
    if any(actual[key] != descriptor[key] for key in required):
        raise AssertionError(f"detector {role} pinned descriptor 불일치")
    return object_name


def upload_detector_pt(path: str | Path, role: str) -> dict[str, Any]:
    path = Path(path)
    file_sha = sha256_file(path)
    artifact_type = {"last": DETECTOR_LAST_TYPE, "best": DETECTOR_BEST_TYPE}[role]
    return upload_create_only(
        BUCKET, path,
        f"{DETECTOR_RESUME_PREFIX}/{role}/{file_sha}.pt",
        artifact_type,
    )


def download_detector_pt(descriptor: dict[str, Any], role: str) -> Path:
    object_name = validate_detector_pt_descriptor(descriptor, role)
    local = LOCAL_META / "detector-resume" / f"{role}-{descriptor['sha256']}.pt"
    download_object(BUCKET, object_name, local, int(descriptor["generation"]))
    if local.stat().st_size != int(descriptor["bytes"]) or sha256_file(local) != descriptor["sha256"]:
        raise AssertionError(f"detector {role} checkpoint SHA/size 오류")
    return local


def validate_epoch_manifest(payload: dict[str, Any], completed: int) -> None:
    expected = {
        "schema_version": "dcc-m3-v4-yolo26s-epoch-manifest-v1",
        "detector_config_sha256": DETECTOR_CONFIG_SHA256,
        "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
        "completed_epochs": completed,
        "target_epochs": DETECTOR_EPOCHS,
        "ultralytics_version": "8.4.121",
    }
    if any(payload.get(key) != value for key, value in expected.items()):
        raise AssertionError("detector epoch manifest/config 불일치")
    validate_detector_pt_descriptor(payload.get("last"), "last")
    validate_detector_pt_descriptor(payload.get("best"), "best")


def latest_detector_epoch(
) -> tuple[dict[str, Any], dict[str, Any]] | None:
    latest = None
    gap_seen = False
    for completed in range(1, DETECTOR_EPOCHS + 1):
        loaded = load_detector_json(
            f"{DETECTOR_RESUME_PREFIX}/epochs/epoch-{completed:03d}.json",
            DETECTOR_EPOCH_TYPE,
        )
        if loaded is None:
            gap_seen = True
            continue
        if gap_seen:
            raise RuntimeError(
                f"detector epoch manifest가 연속하지 않습니다: epoch={completed}"
            )
        validate_epoch_manifest(loaded[0], completed)
        latest = loaded
    return latest


def checkpoint_detector_epoch(trainer: Any) -> None:
    if int(trainer.args.batch) != int(YOLO_BATCH):
        raise RuntimeError(
            "Ultralytics가 요청한 batch를 변경했습니다: "
            f"requested={YOLO_BATCH}, actual={trainer.args.batch}. "
            "YOLO_BATCH를 명시적으로 낮추고 새 run version을 사용하세요."
        )
    completed = int(trainer.epoch) + 1
    if not 1 <= completed <= DETECTOR_EPOCHS:
        raise AssertionError("detector epoch 범위 오류")
    last_path, best_path = Path(trainer.last), Path(trainer.best)
    if not last_path.exists() or not best_path.exists():
        raise RuntimeError("detector last.pt/best.pt 누락")
    payload = {
        "schema_version": "dcc-m3-v4-yolo26s-epoch-manifest-v1",
        "detector_config_sha256": DETECTOR_CONFIG_SHA256,
        "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
        "completed_epochs": completed,
        "target_epochs": DETECTOR_EPOCHS,
        "ultralytics_version": "8.4.121",
        "last": upload_detector_pt(last_path, "last"),
        "best": upload_detector_pt(best_path, "best"),
    }
    upload_json_create_only(
        BUCKET, payload,
        f"{DETECTOR_RESUME_PREFIX}/epochs/epoch-{completed:03d}.json",
        DETECTOR_EPOCH_TYPE,
    )
    print(f"DETECTOR EPOCH {completed} CHECKPOINTED")


def validate_resume_pt(path: Path, manifest: dict[str, Any]) -> None:
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(checkpoint, dict):
        raise AssertionError("detector resume checkpoint 형식 오류")
    if checkpoint.get("version") != "8.4.121":
        raise AssertionError("detector resume ultralytics version 오류")
    if int(checkpoint.get("epoch", -1)) + 1 != int(manifest["completed_epochs"]):
        raise AssertionError("detector resume epoch 오류")
    if any(checkpoint.get(key) is None for key in ["ema", "optimizer", "scaler"]):
        raise AssertionError("detector resume optimizer/EMA/scaler 누락")
    args = checkpoint.get("train_args")
    expected = {
        "data": str(YOLO_DATA_YAML), "epochs": DETECTOR_EPOCHS,
        "imgsz": YOLO_IMGSZ, "batch": YOLO_BATCH, "seed": SEED,
        "pretrained": True, "deterministic": True, "amp": True,
        "cache": False, "fraction": 1.0, "save_period": 1,
    }
    if not isinstance(args, dict) or any(args.get(k) != v for k, v in expected.items()):
        raise AssertionError("detector resume train_args/config 불일치")


def validate_completion(payload: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any]]:
    expected = {
        "schema_version": "dcc-m3-v4-yolo26s-completion-manifest-v1",
        "detector_config_sha256": DETECTOR_CONFIG_SHA256,
        "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
        "completed_epochs": DETECTOR_EPOCHS,
        "target_epochs": DETECTOR_EPOCHS,
    }
    if any(payload.get(key) != value for key, value in expected.items()):
        raise AssertionError("detector completion manifest/config 불일치")
    loaded = load_detector_json(
        f"{DETECTOR_RESUME_PREFIX}/epochs/epoch-{DETECTOR_EPOCHS:03d}.json",
        DETECTOR_EPOCH_TYPE,
    )
    if loaded is None:
        raise AssertionError("detector final epoch manifest 누락")
    epoch_payload, epoch_descriptor = loaded
    validate_epoch_manifest(epoch_payload, DETECTOR_EPOCHS)
    if payload.get("final_epoch_manifest") != epoch_descriptor or payload.get("best") != epoch_payload["best"]:
        raise AssertionError("detector completion chain 불일치")
    return epoch_payload, epoch_descriptor


DETECTOR_COMPLETION_DESCRIPTOR = None
if DETECTOR_MODE == "retrain_v4":
    completed = load_detector_json(DETECTOR_COMPLETION_OBJECT, DETECTOR_COMPLETION_TYPE)
    if completed is not None:
        completion_payload, DETECTOR_COMPLETION_DESCRIPTOR = completed
        final_epoch, _ = validate_completion(completion_payload)
        DETECTOR_DESCRIPTOR = final_epoch["best"]
        detector_best = download_detector_pt(DETECTOR_DESCRIPTOR, "best")
        print("DETECTOR COMPLETE - verified best.pt reused")
    else:
        latest = latest_detector_epoch()
        if latest is None or int(latest[0]["completed_epochs"]) < DETECTOR_EPOCHS:
            if latest is None:
                detector_trainer = YOLO(DETECTOR_BASE_WEIGHTS)
                detector_trainer.add_callback("on_model_save", checkpoint_detector_epoch)
                detector_result = detector_trainer.train(
                    data=str(YOLO_DATA_YAML), epochs=DETECTOR_EPOCHS,
                    imgsz=YOLO_IMGSZ, batch=YOLO_BATCH, device=0, seed=SEED,
                    project=str(LOCAL_RUN / "detector"), name="train", exist_ok=True,
                    pretrained=True, deterministic=True, workers=2,
                    amp=True, cache=False, fraction=1.0, save_period=1,
                    mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0,
                    multi_scale=False, close_mosaic=0,
                    fliplr=0.0, flipud=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
                    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
                    verbose=True,
                )
            else:
                resume_path = download_detector_pt(latest[0]["last"], "last")
                validate_resume_pt(resume_path, latest[0])
                detector_trainer = YOLO(str(resume_path))
                detector_trainer.add_callback("on_model_save", checkpoint_detector_epoch)
                detector_result = detector_trainer.train(
                    resume=True, device=0, workers=2, cache=False,
                    save_period=1, verbose=True,
                )
            del detector_trainer, detector_result
            latest = latest_detector_epoch()
        if latest is None:
            raise RuntimeError("detector final epoch manifest 누락")
        final_epoch, final_epoch_descriptor = latest
        validate_epoch_manifest(final_epoch, DETECTOR_EPOCHS)
        DETECTOR_DESCRIPTOR = final_epoch["best"]
        detector_best = download_detector_pt(DETECTOR_DESCRIPTOR, "best")
        completion_payload = {
            "schema_version": "dcc-m3-v4-yolo26s-completion-manifest-v1",
            "detector_config_sha256": DETECTOR_CONFIG_SHA256,
            "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
            "completed_epochs": DETECTOR_EPOCHS,
            "target_epochs": DETECTOR_EPOCHS,
            "final_epoch_manifest": final_epoch_descriptor,
            "best": DETECTOR_DESCRIPTOR,
        }
        DETECTOR_COMPLETION_DESCRIPTOR = upload_json_create_only(
            BUCKET, completion_payload, DETECTOR_COMPLETION_OBJECT,
            DETECTOR_COMPLETION_TYPE,
        )
        validate_completion(completion_payload)
        print("DETECTOR COMPLETE - completion manifest frozen")
else:
    blob = BUCKET.blob(FROZEN_DETECTOR_OBJECT)
    blob.reload()
    metadata = blob.metadata or {}
    if metadata.get("artifact_type") not in FROZEN_DETECTOR_ALLOWED_ARTIFACT_TYPES:
        raise AssertionError("reuse detector artifact type 오류")
    DETECTOR_DESCRIPTOR = {
        "uri": f"gs://{BUCKET_NAME}/{FROZEN_DETECTOR_OBJECT}",
        "generation": int(blob.generation), "bytes": int(blob.size),
        "sha256": metadata["sha256"], "artifact_type": metadata["artifact_type"],
    }
    detector_best = LOCAL_META / "reused-detector.pt"
    download_object(BUCKET, FROZEN_DETECTOR_OBJECT, detector_best, DETECTOR_DESCRIPTOR["generation"])
if sha256_file(detector_best) != DETECTOR_DESCRIPTOR["sha256"]:
    raise AssertionError("detector checkpoint SHA 오류")
# Ultralytics teardown disables its deterministic process flags; restore the
# frozen ConvNeXt runtime contract before route inference and regression.
enforce_deterministic_runtime()

INFERENCE_RULE = {
    "detector_confidence": YOLO_CONFIDENCE, "max_detections": YOLO_MAX_DET,
    "detector_imgsz": YOLO_IMGSZ, "detector_iou": YOLO_IOU_THRESHOLD,
    "crop_padding": ROUTED_CROP_PADDING,
    "meter_fusion": "confidence_weighted_median",
    "obstacle_fusion": "confidence_weighted_mean",
    "invalid_candidate_policy": "drop_invalid_or_degenerate_then_fallback_if_empty",
    "detector_miss_fallback": "full_frame_regressor",
}
ROUTING_CONTRACT = {
    "schema_version": "dcc-m3-v4-fixed-router-contract-v1",
    "routing_key": "type_from_sample_id_first_character",
    "route_by_type": ROUTE_BY_TYPE,
    "components": ["full_frame", "yolo_roi"],
    "detector_checkpoint_sha256": DETECTOR_DESCRIPTOR["sha256"],
    "detector_training_config_sha256": DETECTOR_CONFIG_SHA256,
    "detector_completion_sha256": (
        DETECTOR_COMPLETION_DESCRIPTOR["sha256"]
        if DETECTOR_COMPLETION_DESCRIPTOR is not None else None
    ),
    "detector_frozen_between_regression_arms": True,
    "inference_rule": INFERENCE_RULE,
}
ROUTING_SHA256 = canonical_sha256(ROUTING_CONTRACT)


def valid_box(box, width, height):
    box = np.asarray(box, dtype=float)
    if box.shape != (4,) or not np.isfinite(box).all():
        return None
    x1, y1, x2, y2 = box
    x1, x2 = np.clip([x1, x2], 0, width)
    y1, y2 = np.clip([y1, y2], 0, height)
    if x2 - x1 < 1 or y2 - y1 < 1:
        return None
    return [x1 / width, y1 / height, x2 / width, y2 / height]


def route_cache_metadata(split_name, frame):
    return {
        "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
        "detector_checkpoint_sha256": DETECTOR_DESCRIPTOR["sha256"],
        "routing_sha256": ROUTING_SHA256,
        "split_name": split_name,
        "rows": len(frame),
        "stem_sha256": canonical_sha256(sorted(frame["stem"].astype(str))),
    }


def validate_route_cache(cache, frame, split_name):
    expected_columns = {
        "stem", "route_component", "boxes_json", "conf_json", "n_candidates",
        "fallback_used", "detector_predictions_materialized", "image_sha256",
        "detector_sha256", "routing_sha256",
    }
    if set(cache.columns) != expected_columns:
        raise AssertionError(
            f"route cache column 오류: {sorted(set(cache.columns) ^ expected_columns)}"
        )
    cache = cache.sort_values("stem", kind="mergesort").reset_index(drop=True)
    if len(cache) != len(frame) or not cache["stem"].is_unique:
        raise AssertionError("route cache row/uniqueness 오류")
    expected_stems = set(frame["stem"].astype(str))
    if set(cache["stem"].astype(str)) != expected_stems:
        raise AssertionError("route cache stem 완전성 오류")
    expected_hashes = (
        frame.set_index("stem")["image_sha256"].astype(str).sort_index()
    )
    observed_hashes = (
        cache.set_index("stem")["image_sha256"].astype(str).sort_index()
    )
    if not observed_hashes.equals(expected_hashes):
        raise AssertionError("route cache image SHA 오류")
    expected_routes = (
        frame.assign(
            route_component=frame["type"].map(ROUTE_BY_TYPE)
        ).set_index("stem")["route_component"].sort_index()
    )
    observed_routes = cache.set_index("stem")["route_component"].sort_index()
    if not observed_routes.equals(expected_routes):
        raise AssertionError("route cache fixed route 오류")
    if not cache["detector_sha256"].astype(str).eq(
        DETECTOR_DESCRIPTOR["sha256"]
    ).all():
        raise AssertionError("route cache detector SHA 오류")
    if not cache["routing_sha256"].astype(str).eq(ROUTING_SHA256).all():
        raise AssertionError("route cache routing SHA 오류")
    predictions_materialized = split_name == "val"
    if not cache["detector_predictions_materialized"].astype(bool).eq(
        predictions_materialized
    ).all():
        raise AssertionError("route cache prediction materialization 오류")
    boxes = cache["boxes_json"].map(json.loads)
    confidences = cache["conf_json"].map(json.loads)
    counts = cache["n_candidates"].astype(int)
    if (
        not boxes.map(len).eq(counts).all()
        or not confidences.map(len).eq(counts).all()
        or counts.lt(0).any()
    ):
        raise AssertionError("route cache candidate count 오류")
    for box_list, confidence_list in zip(boxes, confidences):
        if any(
            len(box) != 4
            or not all(math.isfinite(float(value)) for value in box)
            or not (0.0 <= float(box[0]) < float(box[2]) <= 1.0)
            or not (0.0 <= float(box[1]) < float(box[3]) <= 1.0)
            for box in box_list
        ):
            raise AssertionError("route cache bbox 오류")
        if any(
            not math.isfinite(float(value)) or float(value) <= 0.0
            for value in confidence_list
        ):
            raise AssertionError("route cache confidence 오류")
    expected_fallback = (
        predictions_materialized
        & cache["route_component"].eq("yolo_roi")
        & counts.eq(0)
    )
    if not cache["fallback_used"].astype(bool).eq(expected_fallback).all():
        raise AssertionError("route cache fallback 오류")
    return frame.merge(
        cache, on=["stem", "image_sha256"], validate="one_to_one"
    )


def build_route_cache(split_name, frame):
    cache_object = f"{RUN_PREFIX}/routing_cache/{split_name}.parquet"
    cache_metadata = route_cache_metadata(split_name, frame)
    cache_path = LOCAL_ROUTING / f"{split_name}-{ROUTING_SHA256[:16]}.parquet"
    descriptor = try_remote_descriptor(
        BUCKET,
        cache_object,
        "m3_v4_router_cache",
        cache_metadata,
    )
    if descriptor is not None:
        download_verified_descriptor(
            BUCKET, descriptor, cache_path, cache_metadata
        )
        cache = pd.read_parquet(cache_path)
        routed = validate_route_cache(cache, frame, split_name)
        print("ROUTE CACHE REUSED", split_name, descriptor["uri"])
        return routed, descriptor

    # train은 full=전체 frame, ROI=GT-largest를 사용하므로 detector prediction이 필요 없다.
    detected = {}
    predictions_materialized = split_name == "val"
    if predictions_materialized:
        detector_dir = LOCAL_ROUTING / f"{split_name}-detector"
        detector_dir.mkdir(parents=True, exist_ok=True)
        roi_frame = frame.loc[frame["type"].isin(ROI_TYPES)]
        for row in roi_frame.itertuples(index=False):
            target = detector_dir / f"{row.stem}.jpg"
            if not target.exists():
                os.link(row.image_path, target)
        detector = YOLO(str(detector_best))
        stream = detector.predict(
            source=str(detector_dir), imgsz=YOLO_IMGSZ, batch=YOLO_BATCH,
            conf=YOLO_CONFIDENCE, iou=YOLO_IOU_THRESHOLD, max_det=YOLO_MAX_DET,
            device=0, stream=True, save=False, verbose=False,
        )
        for result in stream:
            stem = Path(result.path).stem
            with Image.open(result.path) as image:
                width, height = image.size
            boxes = result.boxes.xyxy.detach().cpu().numpy() if result.boxes is not None else []
            conf = result.boxes.conf.detach().cpu().numpy() if result.boxes is not None else []
            pairs = [(valid_box(b, width, height), float(c)) for b, c in zip(boxes, conf)]
            detected[stem] = [(b,c) for b,c in pairs if b is not None and math.isfinite(c) and c > 0]
        if set(detected) != set(roi_frame["stem"]):
            raise AssertionError("route detector stem 완전성 오류")
        del detector, stream
        gc.collect()
        torch.cuda.empty_cache()
    records = []
    for row in frame.itertuples(index=False):
        route = ROUTE_BY_TYPE[row.type]
        pairs = detected.get(row.stem, []) if predictions_materialized and route == "yolo_roi" else []
        records.append({
            "stem":row.stem, "route_component":route,
            "boxes_json":json.dumps([x[0] for x in pairs],separators=(",",":")),
            "conf_json":json.dumps([x[1] for x in pairs],separators=(",",":")),
            "n_candidates":len(pairs),
            "fallback_used":bool(predictions_materialized and route=="yolo_roi" and not pairs),
            "detector_predictions_materialized":predictions_materialized,
            "image_sha256":row.image_sha256,
            "detector_sha256":DETECTOR_DESCRIPTOR["sha256"],
            "routing_sha256":ROUTING_SHA256,
        })
    cache = pd.DataFrame(records).sort_values("stem",kind="mergesort")
    routed = validate_route_cache(cache, frame, split_name)
    cache.to_parquet(cache_path,index=False,compression="zstd")
    descriptor = upload_create_only(
        BUCKET,
        cache_path,
        cache_object,
        "m3_v4_router_cache",
        cache_metadata,
    )
    return routed, descriptor


TRAIN_ROUTED, TRAIN_ROUTE_CACHE_DESCRIPTOR = build_route_cache("train", TRAIN_META)
if RUN_MODE == "full":
    VAL_ROUTE_SOURCE = VAL_META
else:
    # stem 정렬 뒤 유형마다 8장을 선택해 smoke에도 8유형과 두 route가 모두 남는다.
    VAL_ROUTE_SOURCE = (
        VAL_META.sort_values(["type", "stem"], kind="mergesort")
        .groupby("type", sort=False, group_keys=False)
        .head(8)
        .sort_values("stem", kind="mergesort")
        .reset_index(drop=True)
    )
    smoke_type_counts = (
        VAL_ROUTE_SOURCE["type"].value_counts().sort_index().astype(int).to_dict()
    )
    expected_smoke_type_counts = {name: 8 for name in TYPE_ORDER}
    if smoke_type_counts != expected_smoke_type_counts:
        raise AssertionError(f"smoke validation type coverage 오류: {smoke_type_counts}")
VAL_ROUTED, VAL_ROUTE_CACHE_DESCRIPTOR = build_route_cache("val", VAL_ROUTE_SOURCE)
EXPECTED_VAL_ROUTE_COUNTS = {"yolo_roi": 23_976, "full_frame": 6_832}
VAL_ROUTE_COUNTS = VAL_ROUTED["route_component"].value_counts().astype(int).to_dict()
if RUN_MODE == "full" and VAL_ROUTE_COUNTS != EXPECTED_VAL_ROUTE_COUNTS:
    raise AssertionError(f"validation route count 오류: {VAL_ROUTE_COUNTS}")


stream materialize train 10 / 177
stream materialize train 20 / 177
stream materialize train 30 / 177
stream materialize train 40 / 177
stream materialize train 50 / 177
stream materialize train 60 / 177
stream materialize train 70 / 177
stream materialize train 80 / 177
stream materialize train 90 / 177
stream materialize train 100 / 177
stream materialize train 110 / 177
stream materialize train 120 / 177
stream materialize train 130 / 177
stream materialize train 140 / 177
stream materialize train 150 / 177
stream materialize train 160 / 177
stream materialize train 170 / 177
stream materialize train 177 / 177
stream materialize val 10 / 175
stream materialize val 20 / 175
stream materialize val 30 / 175
stream materialize val 40 / 175
stream materialize val 50 / 175
stream materialize val 60 / 175
stream materialize val 70 / 175
stream materialize val 80 / 175
stream materialize val 90 / 175
stream materialize val 100 / 175
stream materialize val 110 / 175
stream materialize val 12

## 5. Identical routed datasets for control and multitask arms


In [ ]:
class Letterbox:
    def __init__(self, size):
        self.size = int(size)
    def __call__(self, image):
        image = image.convert("RGB")
        width, height = image.size
        scale = min(self.size / width, self.size / height)
        resized_size = (max(1, round(width * scale)), max(1, round(height * scale)))
        resized = image.resize(resized_size, Image.Resampling.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), (114, 114, 114))
        canvas.paste(resized, ((self.size - resized.width) // 2, (self.size - resized.height) // 2))
        return canvas


def build_transform(size):
    return transforms.Compose([
        transforms.Lambda(Letterbox(size)), transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])


def padded_crop(image, box, jitter=0.0):
    width, height = image.size
    x1, y1, x2, y2 = map(float, box)
    bw, bh = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    if jitter:
        cx += random.uniform(-jitter, jitter) * bw
        cy += random.uniform(-jitter, jitter) * bh
    hw, hh = bw * (0.5 + ROUTED_CROP_PADDING), bh * (0.5 + ROUTED_CROP_PADDING)
    px1, py1 = max(0, math.floor((cx - hw) * width)), max(0, math.floor((cy - hh) * height))
    px2, py2 = min(width, math.ceil((cx + hw) * width)), min(height, math.ceil((cy + hh) * height))
    if px2 <= px1 or py2 <= py1:
        raise ValueError("degenerate crop after padding")
    return image.crop((px1, py1, px2, py2))


class ComponentDataset(Dataset):
    def __init__(self, frame, component, training):
        self.component, self.training, self.rows = component, training, []
        for row in frame.to_dict("records"):
            boxes, conf = json.loads(row["boxes_json"]), json.loads(row["conf_json"])
            if component == "full_frame":
                # 모든 유형을 학습·평가하여 ACEFH detector miss도 같은 checkpoint로 fallback한다.
                self.rows.append((row, None, 1.0))
            elif component == "roi":
                if training:
                    # 원래 ROI 학습 계약: GT largest bbox + padding + 4% jitter.
                    valid_gt = [box for box in row["boxes"] if yolo_line(box) is not None]
                    if row["type"] in ROI_TYPES and valid_gt:
                        largest = max(valid_gt, key=lambda b: (float(b[2])-float(b[0]))*(float(b[3])-float(b[1])))
                        self.rows.append((row, largest, 1.0))
                else:
                    # validation/test 계약: frozen shared detector의 모든 유효 predicted candidate.
                    for box, weight in zip(boxes, conf):
                        self.rows.append((row, box, max(float(weight), 1e-6)))
            else:
                raise ValueError(component)
        self.transform = build_transform(IMAGE_SIZE if component == "full_frame" else ROI_IMAGE_SIZE)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row, box, confidence = self.rows[index]
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        if box is not None:
            image = padded_crop(image, box, ROI_TRAIN_JITTER if self.training else 0.0)
        return (
            self.transform(image), torch.tensor(TYPE_TO_INDEX[row["type"]]),
            torch.tensor(row["target_m"], dtype=torch.float32),
            torch.tensor(row["obstruction_targets"], dtype=torch.float32),
            torch.tensor(row["obstruction_mask"], dtype=torch.float32),
            row["stem"], row["type"], torch.tensor(confidence, dtype=torch.float32),
        )


def component_loader(frame, component, training, seed):
    return DataLoader(
        ComponentDataset(frame, component, training),
        batch_size=BATCH_SIZE if component == "full_frame" else ROI_BATCH_SIZE,
        shuffle=training, generator=torch.Generator().manual_seed(seed),
        num_workers=0, pin_memory=DEVICE.type == "cuda", drop_last=False,
    )


class V4ObstacleRegressor(nn.Module):
    def __init__(self, obstacle_count, pretrained):
        super().__init__()
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        self.backbone = models.convnext_tiny(weights=weights)
        feature_dim = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Identity()
        self.type_embedding = nn.Embedding(len(TYPE_ORDER), TYPE_EMBEDDING_DIM)
        self.shared_head = nn.Sequential(
            nn.Linear(feature_dim + TYPE_EMBEDDING_DIM, 256), nn.GELU(), nn.Dropout(DROPOUT)
        )
        self.regression_head = nn.Linear(256, 1)
        self.obstacle_head = nn.Linear(256, obstacle_count) if obstacle_count else None
        self.register_buffer("type_low", torch.tensor([TYPE_RANGES[x][0] for x in TYPE_ORDER]))
        self.register_buffer("type_high", torch.tensor([TYPE_RANGES[x][1] for x in TYPE_ORDER]))
    def forward(self, images, type_index):
        feature = self.backbone(images)
        shared = self.shared_head(torch.cat([feature, self.type_embedding(type_index)], 1))
        raw = self.regression_head(shared).squeeze(1)
        pred = self.type_low[type_index] + torch.sigmoid(raw) * (self.type_high[type_index] - self.type_low[type_index])
        return pred, self.obstacle_head(shared) if self.obstacle_head is not None else None


def regression_loss(pred, target, type_index):
    low = pred.new_tensor([TYPE_RANGES[x][0] for x in TYPE_ORDER])[type_index]
    high = pred.new_tensor([TYPE_RANGES[x][1] for x in TYPE_ORDER])[type_index]
    return F.smooth_l1_loss((pred-low)/(high-low), (target-low)/(high-low), beta=0.05) + (
        0.20 * torch.mean(((pred-target)/20.0)**2)
    )


def masked_obstacle_loss(logits, target, known, pos_weight):
    loss = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight, reduction="none")
    return (loss * known).sum() / known.sum().clamp_min(1)


positive = np.asarray(dataset_seal["train_obstruction_positive_counts"], float)
known = np.asarray(dataset_seal["train_known_counts"], float)
if len(positive) != 15 or np.any(positive <= 0) or np.any(known < positive):
    raise AssertionError("obstacle train stats 오류")
POS_WEIGHT = torch.tensor(
    np.clip((known-positive)/positive, 1, MAX_POS_WEIGHT).astype(np.float32), device=DEVICE
)


def weighted_median(items):
    items = sorted(items)
    half, running = sum(w for _, w in items) / 2, 0
    for value, weight in items:
        running += weight
        if running >= half:
            return float(value)
    return float(items[-1][0])


def predict_component(model, frame, component, use_obstacle):
    result = defaultdict(list)
    model.eval()
    with torch.inference_mode():
        for batch_i, batch in enumerate(component_loader(frame, component, False, SEED)):
            if MAX_VAL_BATCHES is not None and batch_i >= MAX_VAL_BATCHES:
                break
            images, type_index, _, _, _, stems, _, confidence = batch
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type=="cuda"):
                pred, logits = model(images.to(DEVICE), type_index.to(DEVICE))
            probs = torch.sigmoid(logits).cpu().numpy() if use_obstacle else None
            for i, (stem, value, weight) in enumerate(zip(stems, pred.float().cpu().numpy(), confidence.numpy())):
                result[str(stem)].append((
                    float(value), float(weight), probs[i].astype(float).tolist() if probs is not None else []
                ))
    return result


def metric_payload(frame, include_classifier):
    error = frame["pred_m"].to_numpy(float) - frame["target_m"].to_numpy(float)
    by_type = {}
    for type_name in TYPE_ORDER:
        part = frame[frame["type"] == type_name]
        e = part["pred_m"].to_numpy(float) - part["target_m"].to_numpy(float)
        by_type[type_name] = {"n": len(part), "rmse_m": float(np.sqrt(np.mean(e**2)))}
    target = np.stack(frame["obstruction_targets"])
    mask = np.stack(frame["obstruction_mask"])
    obstacle = {}
    for i, label in enumerate(OBSTACLE_ORDER):
        selected = (target[:, i] == 1) & (mask[:, i] == 1)
        if selected.any():
            obstacle[label] = {
                "n": int(selected.sum()), "rmse_m": float(np.sqrt(np.mean(error[selected]**2)))
            }
    thresholds = []
    if include_classifier:
        prob = np.stack(frame["obstacle_probabilities"])
        for i in range(15):
            valid = mask[:, i].astype(bool)
            y, p = target[valid, i], prob[valid, i]
            scores = []
            for threshold in np.linspace(.05, .95, 19):
                q = p >= threshold
                tp, fp, fn = ((y==1)&q).sum(), ((y==0)&q).sum(), ((y==1)&~q).sum()
                scores.append((2*tp/max(1, 2*tp+fp+fn), -float(threshold)))
            thresholds.append(-max(scores)[1])
    overall = float(np.sqrt(np.mean(error**2)))
    return {
        "overall_rmse_m": overall, "overall_mae_m": float(np.mean(np.abs(error))),
        "worst_type_rmse_m": max(x["rmse_m"] for x in by_type.values()),
        "worst_obstacle_rmse_m": max([x["rmse_m"] for x in obstacle.values()] or [overall]),
        "by_type": by_type, "obstacle_slices": obstacle,
        "obstacle_classifier": {"validation_selected_thresholds": thresholds},
    }


def evaluate_pair(models_by_component, use_obstacle):
    full = predict_component(models_by_component["full_frame"], VAL_ROUTED, "full_frame", use_obstacle)
    roi = predict_component(models_by_component["roi"], VAL_ROUTED, "roi", use_obstacle)
    records = []
    for row in VAL_ROUTED.itertuples(index=False):
        if row.route_component == "yolo_roi" and not row.fallback_used:
            values = roi.get(row.stem, [])
            if not values:
                if MAX_VAL_BATCHES is not None: continue
                raise AssertionError("ROI prediction 누락")
            pred = weighted_median([(v, w) for v, w, _ in values])
            if use_obstacle:
                weights = np.asarray([w for _, w, _ in values])
                probability = np.average(np.asarray([p for _, _, p in values]), 0, weights=weights).tolist()
            else:
                probability = []
        else:
            values = full.get(row.stem, [])
            if len(values) != 1:
                if MAX_VAL_BATCHES is not None: continue
                raise AssertionError("full prediction 누락/중복")
            pred, _, probability = values[0]
        records.append({
            "stem": row.stem, "type": row.type, "target_m": row.target_m, "pred_m": pred,
            "obstruction_targets": row.obstruction_targets,
            "obstruction_mask": row.obstruction_mask,
            "obstacle_probabilities": probability,
            "route_component": row.route_component,
            "fallback_used": bool(row.fallback_used), "n_candidates": int(row.n_candidates),
        })
    output = pd.DataFrame(records).sort_values("stem", kind="mergesort")
    if RUN_MODE == "full" and len(output) != EXPECTED_VAL_N:
        raise AssertionError("validation routed count 오류")
    return output, metric_payload(output, use_obstacle)


def architecture_payload(candidate, component):
    return {
        "schema_version": "dcc-m3-v4-obstacle-regressor-architecture-v1",
        "model_class": "V4ObstacleRegressor",
        "input_contract": ["images", "type_index"], "candidate_name": candidate,
        "component": component, "backbone": "convnext_tiny",
        "image_size": IMAGE_SIZE if component == "full_frame" else ROI_IMAGE_SIZE,
        "preprocessing": {"resize": "letterbox", "fill_rgb": [114,114,114], "mean": [0.485,0.456,0.406], "std": [0.229,0.224,0.225]},
        "type_embedding_dim": TYPE_EMBEDDING_DIM, "hidden_dim": 256, "dropout": DROPOUT,
        "forward_signature": "(images,type_index)->(pred_m,obstacle_logits_or_none)",
        "type_order": TYPE_ORDER, "type_ranges": {k:list(v) for k,v in TYPE_RANGES.items()},
        "use_obstacle_head": candidate == "multitask",
        "obstacle_vocabulary": OBSTACLE_ORDER if candidate == "multitask" else [],
    }


RUN_CONFIG = {
    "schema_version": "dcc-m3-v4-router-training-config-v3",
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "dataset_version": DATASET_VERSION,
    "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
    "detector_config": DETECTOR_CONFIG,
    "detector": DETECTOR_DESCRIPTOR,
    "detector_completion": DETECTOR_COMPLETION_DESCRIPTOR,
    "routing_contract": ROUTING_CONTRACT,
    "routing_sha256": ROUTING_SHA256,
    "route_cache": {
        "train": TRAIN_ROUTE_CACHE_DESCRIPTOR,
        "val": VAL_ROUTE_CACHE_DESCRIPTOR,
    },
    "candidates": CANDIDATE_NAMES,
    "components": COMPONENT_NAMES,
    "epochs": EPOCHS,
    "optimizer": {
        "name": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    },
    "scheduler": {"name": "CosineAnnealingLR", "t_max": EPOCHS},
    "amp": {"enabled": DEVICE.type == "cuda", "dtype": "float16"},
    "seed": SEED,
    "batch_size": {"full_frame": BATCH_SIZE, "roi": ROI_BATCH_SIZE},
    "image_size": {"full_frame": IMAGE_SIZE, "roi": ROI_IMAGE_SIZE},
    "data_loader": {
        "num_workers": 0,
        "drop_last": False,
        "shuffle_seed_rule": "seed + epoch * 10 + component_index",
    },
    "model": {
        "backbone": "convnext_tiny",
        "pretrained_weights": "ConvNeXt_Tiny_Weights.DEFAULT",
        "type_embedding_dim": TYPE_EMBEDDING_DIM,
        "hidden_dim": 256,
        "dropout": DROPOUT,
    },
    "loss": {
        "regression": "normalized_smooth_l1_beta_0.05_plus_0.20_scaled_mse",
        "obstacle": "known_masked_bce_with_logits",
        "obstacle_loss_weight": OBSTACLE_LOSS_WEIGHT,
        "max_pos_weight": MAX_POS_WEIGHT,
    },
    "roi_training": {
        "source": "largest_valid_ground_truth_box",
        "crop_padding": ROUTED_CROP_PADDING,
        "jitter": ROI_TRAIN_JITTER,
    },
    "augment": AUGMENT,
    "max_train_batches": MAX_TRAIN_BATCHES,
    "max_val_batches": MAX_VAL_BATCHES,
    "minimum_type_obstacle_slice_n": MIN_TYPE_OBSTACLE_SLICE_N,
    "resume_contract": {
        "schema_version": "dcc-m3-v4-router-training-checkpoint-v2",
        "granularity": "completed_epoch",
        "candidate_completion_schema": "dcc-m3-v4-router-candidate-completion-v1",
    },
    "runtime_contract": RUNTIME_CONTRACT,
    "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
}
RUN_CONFIG_SHA256 = canonical_sha256(RUN_CONFIG)

TRAINING_CHECKPOINT_SCHEMA = "dcc-m3-v4-router-training-checkpoint-v2"
TRAINING_CHECKPOINT_ARTIFACT = "m3_v4_router_training_checkpoint_v2"
CANDIDATE_COMPLETION_SCHEMA = "dcc-m3-v4-router-candidate-completion-v1"
CANDIDATE_COMPLETION_ARTIFACT = "m3_v4_router_candidate_completion"


def training_score(row: dict[str, Any]) -> tuple[float, float, float, int]:
    return (
        float(row["val_overall_rmse_m"]),
        float(row["val_worst_type_rmse_m"]),
        float(row["val_worst_obstacle_rmse_m"]),
        int(row["epoch"]),
    )


def model_state_sha256(state_dict: dict[str, torch.Tensor]) -> str:
    digest = hashlib.sha256()
    for key, value in sorted(state_dict.items()):
        if not torch.is_tensor(value):
            raise TypeError(f"state_dict tensor가 아닙니다: {key}")
        tensor = value.detach().cpu().contiguous()
        digest.update(key.encode("utf-8"))
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(json.dumps(list(tensor.shape), separators=(",", ":")).encode("ascii"))
        digest.update(tensor.numpy().tobytes(order="C"))
    return digest.hexdigest()


def capture_rng_state() -> dict[str, Any]:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all(),
    }


def restore_rng_state(state: dict[str, Any]) -> None:
    required = {"python", "numpy", "torch_cpu", "torch_cuda"}
    if set(state) != required:
        raise AssertionError(f"RNG state key 오류: {set(state)}")
    cuda_states = state["torch_cuda"]
    if len(cuda_states) != torch.cuda.device_count():
        raise RuntimeError("checkpoint CUDA RNG device 수가 현재 runtime과 다릅니다")
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    torch.cuda.set_rng_state_all(cuda_states)


def optimizer_state_to_device(optimizer_object: AdamW) -> None:
    for state in optimizer_object.state.values():
        for key, value in list(state.items()):
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)


def checkpoint_object_name(candidate_name: str, epoch: int) -> str:
    return (
        f"{RUN_PREFIX}/{candidate_name}/checkpoints/"
        f"epoch-{int(epoch):03d}.pt"
    )


def checkpoint_metadata(candidate_name: str, epoch: int) -> dict[str, Any]:
    return {
        "run_config_sha256": RUN_CONFIG_SHA256,
        "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
        "candidate_name": candidate_name,
        "completed_epoch": int(epoch),
    }


def validate_history(history: list[dict[str, Any]], completed_epoch: int) -> None:
    expected_epochs = list(range(1, int(completed_epoch) + 1))
    actual_epochs = [int(row.get("epoch", -1)) for row in history]
    if actual_epochs != expected_epochs:
        raise AssertionError(
            f"checkpoint history epoch 불연속: expected={expected_epochs}, actual={actual_epochs}"
        )
    for row in history:
        values = training_score(row)[:3]
        if not all(math.isfinite(value) for value in values):
            raise AssertionError(f"checkpoint history metric finite 오류: {row}")


def validate_training_checkpoint(
    checkpoint: dict[str, Any],
    candidate_name: str,
    completed_epoch: int,
) -> None:
    expected_scalars = {
        "schema_version": TRAINING_CHECKPOINT_SCHEMA,
        "run_id": RUN_ID,
        "run_mode": RUN_MODE,
        "candidate_name": candidate_name,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
        "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
        "detector_checkpoint_sha256": DETECTOR_DESCRIPTOR["sha256"],
        "routing_sha256": ROUTING_SHA256,
        "completed_epoch": int(completed_epoch),
        "next_epoch": int(completed_epoch) + 1,
        "resume_granularity": "completed_epoch",
    }
    for key, expected in expected_scalars.items():
        if checkpoint.get(key) != expected:
            raise AssertionError(
                f"checkpoint {key} 불일치: expected={expected}, actual={checkpoint.get(key)}"
            )
    if checkpoint.get("component_order") != COMPONENT_NAMES:
        raise AssertionError("checkpoint component 순서 오류")
    for key in ["models", "optimizers", "schedulers", "scalers"]:
        if set(checkpoint.get(key, {})) != set(COMPONENT_NAMES):
            raise AssertionError(f"checkpoint {key} component 집합 오류")
    fingerprints = checkpoint.get("initialization_fingerprints", {})
    if set(fingerprints) != set(COMPONENT_NAMES):
        raise AssertionError("checkpoint initialization fingerprint 오류")
    history = checkpoint.get("history")
    if not isinstance(history, list):
        raise AssertionError("checkpoint history 형식 오류")
    validate_history(history, completed_epoch)
    if checkpoint.get("best_so_far") != min(history, key=training_score):
        raise AssertionError("checkpoint best_so_far 불일치")
    if not isinstance(checkpoint.get("rng_state"), dict):
        raise AssertionError("checkpoint RNG state 누락")


def probe_candidate_checkpoints(
    candidate_name: str,
    candidate_dir: Path,
) -> tuple[dict[int, dict[str, Any]], dict[str, Any] | None]:
    descriptors: dict[int, dict[str, Any]] = {}
    gap_seen = False
    for epoch in range(1, EPOCHS + 1):
        descriptor = try_remote_descriptor(
            BUCKET,
            checkpoint_object_name(candidate_name, epoch),
            TRAINING_CHECKPOINT_ARTIFACT,
            checkpoint_metadata(candidate_name, epoch),
        )
        if descriptor is None:
            gap_seen = True
            continue
        if gap_seen:
            raise RuntimeError(
                f"{candidate_name} checkpoint epoch가 연속하지 않습니다: epoch={epoch}"
            )
        descriptors[epoch] = descriptor
    if not descriptors:
        return descriptors, None
    completed_epoch = max(descriptors)
    checkpoint = load_remote_torch(
        BUCKET,
        descriptors[completed_epoch],
        candidate_dir / f"epoch-{completed_epoch:03d}.pt",
        checkpoint_metadata(candidate_name, completed_epoch),
    )
    validate_training_checkpoint(checkpoint, candidate_name, completed_epoch)
    return descriptors, checkpoint


def completion_object_name(candidate_name: str) -> str:
    return f"{RUN_PREFIX}/{candidate_name}/candidate_complete.json"


def validate_candidate_result(
    result: dict[str, Any],
    candidate_name: str,
) -> None:
    if result.get("candidate_name") != candidate_name:
        raise AssertionError("candidate result 이름 오류")
    if result.get("run_config_sha256") != RUN_CONFIG_SHA256:
        raise AssertionError("candidate result run config 오류")
    if result.get("runtime_contract_sha256") != RUNTIME_CONTRACT_SHA256:
        raise AssertionError("candidate result runtime contract 오류")
    if result.get("routing_sha256") != ROUTING_SHA256:
        raise AssertionError("candidate result routing contract 오류")
    history = result.get("history")
    if not isinstance(history, list):
        raise AssertionError("candidate result history 누락")
    validate_history(history, EPOCHS)
    expected_best = int(min(history, key=training_score)["epoch"])
    if int(result.get("best_epoch", -1)) != expected_best:
        raise AssertionError("candidate result best epoch 오류")
    if set(result.get("initialization_fingerprints", {})) != set(COMPONENT_NAMES):
        raise AssertionError("candidate result initialization fingerprint 오류")
    if result.get("validation_route_counts") != VAL_ROUTE_COUNTS:
        raise AssertionError("candidate result validation route count 오류")
    expected_validation_rows = EXPECTED_VAL_N if RUN_MODE == "full" else len(VAL_ROUTED)
    if int(result.get("validation_rows", -1)) != expected_validation_rows:
        raise AssertionError("candidate result validation row count 오류")
    if RUN_MODE == "full" and (
        result.get("validation_stem_sha256") != dataset_seal["validation_stem_sha256"]
        or result.get("validation_target_sha256") != dataset_seal["validation_target_sha256"]
        or result.get("validation_obstruction_sha256")
        != dataset_seal["validation_obstruction_sha256"]
    ):
        raise AssertionError("candidate result validation hash 오류")
    metrics = result.get("metrics", {})
    metric_values = [
        metrics.get("overall_rmse_m"),
        metrics.get("worst_type_rmse_m"),
        metrics.get("worst_obstacle_rmse_m"),
    ]
    if not all(value is not None and math.isfinite(float(value)) for value in metric_values):
        raise AssertionError("candidate result metric finite 오류")
    components = result.get("components", {})
    if set(components) != {
        "yolo_detector", "full_frame_regressor", "roi_regressor"
    }:
        raise AssertionError("candidate result component 집합 오류")
    if canonical_sha256(components) != result.get("checkpoint_bundle_sha256"):
        raise AssertionError("candidate result checkpoint bundle hash 오류")
    validate_pinned_descriptor(BUCKET, result["prediction"])
    validate_pinned_descriptor(BUCKET, components["yolo_detector"])
    for key in ["full_frame_regressor", "roi_regressor"]:
        validate_pinned_descriptor(BUCKET, components[key]["checkpoint"])


def load_candidate_completion(
    candidate_name: str,
    candidate_dir: Path,
) -> tuple[dict[str, Any], dict[str, Any]] | None:
    required_metadata = {
        "run_config_sha256": RUN_CONFIG_SHA256,
        "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
        "candidate_name": candidate_name,
    }
    descriptor = try_remote_descriptor(
        BUCKET,
        completion_object_name(candidate_name),
        CANDIDATE_COMPLETION_ARTIFACT,
        required_metadata,
    )
    if descriptor is None:
        return None
    payload = load_remote_json(
        BUCKET,
        descriptor,
        candidate_dir / "candidate_complete.json",
        required_metadata,
    )
    expected = {
        "schema_version": CANDIDATE_COMPLETION_SCHEMA,
        "status": "CANDIDATE_COMPLETE",
        "run_id": RUN_ID,
        "run_mode": RUN_MODE,
        "candidate_name": candidate_name,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
        "completed_epoch": EPOCHS,
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise AssertionError(f"candidate completion {key} 오류")
    assertions = payload.get("assertions", {})
    if not assertions or not all(assertions.values()):
        raise AssertionError("candidate completion assertion 오류")
    checkpoint_descriptors = payload.get("epoch_checkpoints", [])
    if len(checkpoint_descriptors) != EPOCHS:
        raise AssertionError("candidate completion checkpoint 개수 오류")
    for epoch, checkpoint_descriptor in enumerate(checkpoint_descriptors, 1):
        validate_pinned_descriptor(
            BUCKET,
            checkpoint_descriptor,
            checkpoint_metadata(candidate_name, epoch),
        )
    validate_pinned_descriptor(
        BUCKET,
        payload["best_checkpoint"],
        checkpoint_metadata(candidate_name, int(payload["best_epoch"])),
    )
    validate_pinned_descriptor(
        BUCKET,
        payload["final_checkpoint"],
        checkpoint_metadata(candidate_name, EPOCHS),
    )
    result = payload.get("candidate_result")
    if not isinstance(result, dict):
        raise AssertionError("candidate completion result 누락")
    validate_candidate_result(result, candidate_name)
    return result, descriptor


candidate_results_by_name: dict[str, dict[str, Any]] = {}
candidate_completion_descriptors: dict[str, dict[str, Any]] = {}
INITIALIZATION_REFERENCE: dict[str, str] = {}

for candidate_name in CANDIDATE_NAMES:
    enforce_deterministic_runtime()
    use_obstacle = candidate_name == "multitask"
    candidate_dir = LOCAL_RUN / candidate_name
    candidate_dir.mkdir(parents=True, exist_ok=True)

    completed = load_candidate_completion(candidate_name, candidate_dir)
    if completed is not None:
        completed_result, completion_descriptor = completed
        for component, fingerprint in completed_result[
            "initialization_fingerprints"
        ].items():
            if (
                component in INITIALIZATION_REFERENCE
                and INITIALIZATION_REFERENCE[component] != fingerprint
            ):
                raise AssertionError(
                    f"control/multitask shared initialization mismatch: {component}"
                )
            INITIALIZATION_REFERENCE.setdefault(component, fingerprint)
        candidate_results_by_name[candidate_name] = completed_result
        candidate_completion_descriptors[candidate_name] = completion_descriptor
        print(
            "CANDIDATE COMPLETE REUSED",
            candidate_name,
            completion_descriptor["uri"],
        )
        continue

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    checkpoint_descriptors, resume_checkpoint = probe_candidate_checkpoints(
        candidate_name, candidate_dir
    )
    component_models: dict[str, V4ObstacleRegressor] = {}
    candidate_initialization_fingerprints: dict[str, str] = {}
    for component_index, component in enumerate(COMPONENT_NAMES):
        component_seed = SEED + component_index
        random.seed(component_seed)
        np.random.seed(component_seed)
        torch.manual_seed(component_seed)
        torch.cuda.manual_seed_all(component_seed)
        model = V4ObstacleRegressor(
            15 if use_obstacle else 0,
            pretrained=resume_checkpoint is None,
        ).to(DEVICE)
        if resume_checkpoint is None:
            digest = hashlib.sha256()
            for key, value in sorted(model.state_dict().items()):
                if key.startswith("obstacle_head."):
                    continue
                digest.update(key.encode("utf-8"))
                digest.update(value.detach().cpu().contiguous().numpy().tobytes())
            fingerprint = digest.hexdigest()
        else:
            fingerprint = resume_checkpoint["initialization_fingerprints"][component]
        if (
            component in INITIALIZATION_REFERENCE
            and INITIALIZATION_REFERENCE[component] != fingerprint
        ):
            raise AssertionError(
                f"control/multitask shared initialization mismatch: {component}"
            )
        INITIALIZATION_REFERENCE.setdefault(component, fingerprint)
        candidate_initialization_fingerprints[component] = fingerprint
        component_models[component] = model

    optimizer = {
        component: AdamW(
            component_models[component].parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
        for component in COMPONENT_NAMES
    }
    scheduler = {
        component: CosineAnnealingLR(
            optimizer[component], T_max=max(1, EPOCHS)
        )
        for component in COMPONENT_NAMES
    }
    scaler = {
        component: torch.amp.GradScaler(
            "cuda", enabled=DEVICE.type == "cuda"
        )
        for component in COMPONENT_NAMES
    }

    if resume_checkpoint is None:
        history: list[dict[str, Any]] = []
        start_epoch = 1
    else:
        for component in COMPONENT_NAMES:
            component_models[component].load_state_dict(
                resume_checkpoint["models"][component]
            )
            optimizer[component].load_state_dict(
                resume_checkpoint["optimizers"][component]
            )
            optimizer_state_to_device(optimizer[component])
            scheduler[component].load_state_dict(
                resume_checkpoint["schedulers"][component]
            )
            scaler[component].load_state_dict(
                resume_checkpoint["scalers"][component]
            )
        history = [dict(row) for row in resume_checkpoint["history"]]
        start_epoch = int(resume_checkpoint["next_epoch"])
        restore_rng_state(resume_checkpoint["rng_state"])
        print(
            "CANDIDATE RESUMED",
            candidate_name,
            f"completed_epoch={start_epoch - 1}",
            f"next_epoch={start_epoch}",
        )

    for epoch in range(start_epoch, EPOCHS + 1):
        row: dict[str, Any] = {"epoch": epoch}
        for component_index, component in enumerate(COMPONENT_NAMES):
            model = component_models[component]
            seen, total = 0, 0.0
            model.train()
            loader_seed = SEED + epoch * 10 + component_index
            for batch_index, batch in enumerate(
                component_loader(
                    TRAIN_ROUTED, component, True, loader_seed
                )
            ):
                if (
                    MAX_TRAIN_BATCHES is not None
                    and batch_index >= MAX_TRAIN_BATCHES
                ):
                    break
                images, type_index, target, obs_target, obs_mask, *_ = batch
                images = images.to(DEVICE)
                type_index = type_index.to(DEVICE)
                target = target.to(DEVICE)
                obs_target = obs_target.to(DEVICE)
                obs_mask = obs_mask.to(DEVICE)
                optimizer[component].zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=DEVICE.type == "cuda",
                ):
                    pred, logits = model(images, type_index)
                    regression = regression_loss(pred, target, type_index)
                    obstacle = (
                        masked_obstacle_loss(
                            logits, obs_target, obs_mask, POS_WEIGHT
                        )
                        if use_obstacle
                        else regression.new_zeros(())
                    )
                    loss = regression + (
                        OBSTACLE_LOSS_WEIGHT * obstacle
                        if use_obstacle
                        else 0
                    )
                scaler[component].scale(loss).backward()
                scaler[component].step(optimizer[component])
                scaler[component].update()
                seen += len(target)
                total += float(loss.detach()) * len(target)
            scheduler[component].step()
            row[f"{component}_rows"] = seen
            row[f"{component}_loss"] = total / max(1, seen)

        _, val_metrics = evaluate_pair(component_models, use_obstacle)
        row.update(
            {
                "val_overall_rmse_m": val_metrics["overall_rmse_m"],
                "val_worst_type_rmse_m": val_metrics["worst_type_rmse_m"],
                "val_worst_obstacle_rmse_m": val_metrics[
                    "worst_obstacle_rmse_m"
                ],
            }
        )
        history.append(row)
        validate_history(history, epoch)
        checkpoint = {
            "schema_version": TRAINING_CHECKPOINT_SCHEMA,
            "run_id": RUN_ID,
            "run_mode": RUN_MODE,
            "candidate_name": candidate_name,
            "run_config_sha256": RUN_CONFIG_SHA256,
            "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
            "dataset_seal_sha256": DATASET_SEAL_DESCRIPTOR["sha256"],
            "detector_checkpoint_sha256": DETECTOR_DESCRIPTOR["sha256"],
            "routing_sha256": ROUTING_SHA256,
            "resume_granularity": "completed_epoch",
            "completed_epoch": epoch,
            "next_epoch": epoch + 1,
            "component_order": COMPONENT_NAMES,
            "history": [dict(item) for item in history],
            "best_so_far": dict(min(history, key=training_score)),
            "initialization_fingerprints": (
                candidate_initialization_fingerprints
            ),
            "models": {
                component: component_models[component].state_dict()
                for component in COMPONENT_NAMES
            },
            "optimizers": {
                component: optimizer[component].state_dict()
                for component in COMPONENT_NAMES
            },
            "schedulers": {
                component: scheduler[component].state_dict()
                for component in COMPONENT_NAMES
            },
            "scalers": {
                component: scaler[component].state_dict()
                for component in COMPONENT_NAMES
            },
            "rng_state": capture_rng_state(),
        }
        validate_training_checkpoint(checkpoint, candidate_name, epoch)
        checkpoint_path = candidate_dir / f"epoch-{epoch:03d}.pt"
        torch.save(checkpoint, checkpoint_path)
        checkpoint_descriptor = upload_create_only(
            BUCKET,
            checkpoint_path,
            checkpoint_object_name(candidate_name, epoch),
            TRAINING_CHECKPOINT_ARTIFACT,
            checkpoint_metadata(candidate_name, epoch),
        )
        checkpoint_descriptors[epoch] = checkpoint_descriptor
        checkpoint_path.unlink(missing_ok=True)
        print(candidate_name, row)

    if len(checkpoint_descriptors) != EPOCHS:
        raise AssertionError(
            f"{candidate_name} 완료 checkpoint 개수 오류: "
            f"{len(checkpoint_descriptors)}"
        )
    validate_history(history, EPOCHS)
    best_epoch = int(min(history, key=training_score)["epoch"])
    best_descriptor = checkpoint_descriptors[best_epoch]
    final_descriptor = checkpoint_descriptors[EPOCHS]
    best = load_remote_torch(
        BUCKET,
        best_descriptor,
        candidate_dir / f"best-epoch-{best_epoch:03d}.pt",
        checkpoint_metadata(candidate_name, best_epoch),
    )
    validate_training_checkpoint(best, candidate_name, best_epoch)
    for component in COMPONENT_NAMES:
        component_models[component].load_state_dict(best["models"][component])

    prediction_object = (
        f"{RUN_PREFIX}/{candidate_name}/validation_predictions.parquet"
    )
    prediction_metadata = {
        "run_config_sha256": RUN_CONFIG_SHA256,
        "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
        "candidate_name": candidate_name,
        "best_epoch": best_epoch,
        "best_checkpoint_sha256": best_descriptor["sha256"],
    }
    pred_descriptor = try_remote_descriptor(
        BUCKET,
        prediction_object,
        "m3_v4_router_validation_predictions",
        prediction_metadata,
    )
    pred_path = candidate_dir / "validation_predictions.parquet"
    if pred_descriptor is None:
        prediction_frame, metrics = evaluate_pair(
            component_models, use_obstacle
        )
    else:
        download_verified_descriptor(
            BUCKET, pred_descriptor, pred_path, prediction_metadata
        )
        prediction_frame = pd.read_parquet(pred_path)
        for column in [
            "obstruction_targets",
            "obstruction_mask",
            "obstacle_probabilities",
        ]:
            prediction_frame[column] = prediction_frame[column].map(json.loads)
        prediction_frame = prediction_frame.sort_values(
            "stem", kind="mergesort"
        ).reset_index(drop=True)
        metrics = metric_payload(prediction_frame, use_obstacle)
        print(
            "VALIDATION PREDICTIONS REUSED",
            candidate_name,
            pred_descriptor["uri"],
        )

    stem_sha = canonical_sha256(sorted(prediction_frame["stem"]))
    target_sha = canonical_target_sha256(prediction_frame)
    obstacle_sha = canonical_obstruction_sha256(prediction_frame)
    if RUN_MODE == "full" and (
        stem_sha != dataset_seal["validation_stem_sha256"]
        or target_sha != dataset_seal["validation_target_sha256"]
        or obstacle_sha != dataset_seal["validation_obstruction_sha256"]
    ):
        raise AssertionError("validation contract hash 오류")
    if len(prediction_frame) != (
        EXPECTED_VAL_N if RUN_MODE == "full" else len(VAL_ROUTED)
    ):
        raise AssertionError("validation prediction row count 오류")
    if pred_descriptor is None:
        serial = prediction_frame.copy()
        for column in [
            "obstruction_targets",
            "obstruction_mask",
            "obstacle_probabilities",
        ]:
            serial[column] = serial[column].map(
                lambda value: json.dumps(value, separators=(",", ":"))
            )
        serial.to_parquet(pred_path, index=False, compression="zstd")
        pred_descriptor = upload_create_only(
            BUCKET,
            pred_path,
            prediction_object,
            "m3_v4_router_validation_predictions",
            prediction_metadata,
        )

    components = {"yolo_detector": DETECTOR_DESCRIPTOR}
    for component in COMPONENT_NAMES:
        key = (
            "full_frame_regressor"
            if component == "full_frame"
            else "roi_regressor"
        )
        architecture = architecture_payload(candidate_name, component)
        architecture_fingerprint = canonical_sha256(architecture)
        model_id = (
            f"v4-{candidate_name.replace('_', '-')}-"
            f"{component.replace('_', '-')}-convnext-tiny"
        )
        state_sha256 = model_state_sha256(
            component_models[component].state_dict()
        )
        deployment_object = (
            f"{RUN_PREFIX}/{candidate_name}/deployment/{component}.pt"
        )
        deployment_metadata = {
            "run_config_sha256": RUN_CONFIG_SHA256,
            "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
            "candidate_name": candidate_name,
            "component": component,
            "best_epoch": best_epoch,
            "best_checkpoint_sha256": best_descriptor["sha256"],
            "architecture_fingerprint": architecture_fingerprint,
            "model_state_sha256": state_sha256,
        }
        descriptor = try_remote_descriptor(
            BUCKET,
            deployment_object,
            "m3_v4_router_regressor_checkpoint",
            deployment_metadata,
        )
        if descriptor is None:
            payload = {
                "schema_version": "dcc-m3-v4-router-regressor-checkpoint-v1",
                "component": key,
                "candidate_name": candidate_name,
                "model_id": model_id,
                "use_obstacle_head": use_obstacle,
                "architecture_payload": architecture,
                "architecture_fingerprint": architecture_fingerprint,
                "run_config_sha256": RUN_CONFIG_SHA256,
                "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
                "best_epoch": best_epoch,
                "best_checkpoint_sha256": best_descriptor["sha256"],
                "model_state_sha256": state_sha256,
                "model": component_models[component].state_dict(),
            }
            deployment_path = candidate_dir / f"{component}-deployment.pt"
            torch.save(payload, deployment_path)
            descriptor = upload_create_only(
                BUCKET,
                deployment_path,
                deployment_object,
                "m3_v4_router_regressor_checkpoint",
                deployment_metadata,
            )
        else:
            print(
                "DEPLOYMENT CHECKPOINT REUSED",
                candidate_name,
                component,
                descriptor["uri"],
            )
        components[key] = {
            "model_id": model_id,
            "component": key,
            "image_size": architecture["image_size"],
            "use_obstacle_head": use_obstacle,
            "architecture_payload": architecture,
            "architecture_fingerprint": architecture_fingerprint,
            "checkpoint": descriptor,
        }

    candidate_result = {
        "model_id": f"v4-fixed-router-{candidate_name.replace('_', '-')}",
        "candidate_name": candidate_name,
        "kind": "type_router",
        "task_kind": (
            "multitask_regressor"
            if use_obstacle
            else "single_task_regressor"
        ),
        "run_config_sha256": RUN_CONFIG_SHA256,
        "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
        "route_by_type": ROUTE_BY_TYPE,
        "routing_contract": ROUTING_CONTRACT,
        "routing_sha256": ROUTING_SHA256,
        "inference_rule": INFERENCE_RULE,
        "components": components,
        "checkpoint_bundle_sha256": canonical_sha256(components),
        "use_obstacle_head": use_obstacle,
        "obstacle_vocabulary": OBSTACLE_ORDER if use_obstacle else [],
        "obstacle_thresholds": (
            metrics["obstacle_classifier"]["validation_selected_thresholds"]
            if use_obstacle
            else []
        ),
        "metrics": metrics,
        "validation_rows": len(prediction_frame),
        "validation_stem_sha256": stem_sha,
        "validation_target_sha256": target_sha,
        "validation_obstruction_sha256": obstacle_sha,
        "validation_route_counts": VAL_ROUTE_COUNTS,
        "validation_fallback_count": int(
            prediction_frame["fallback_used"].sum()
        ),
        "prediction": pred_descriptor,
        "best_epoch": best_epoch,
        "best_checkpoint": best_descriptor,
        "final_checkpoint": final_descriptor,
        "history": history,
        "initialization_fingerprints": (
            candidate_initialization_fingerprints
        ),
    }
    validate_candidate_result(candidate_result, candidate_name)
    completion_assertions = {
        "all_epochs_present": len(checkpoint_descriptors) == EPOCHS,
        "history_complete": [row["epoch"] for row in history]
        == list(range(1, EPOCHS + 1)),
        "best_checkpoint_pinned": best_descriptor
        == checkpoint_descriptors[best_epoch],
        "final_checkpoint_pinned": final_descriptor
        == checkpoint_descriptors[EPOCHS],
        "validation_rows_exact": len(prediction_frame)
        == (EXPECTED_VAL_N if RUN_MODE == "full" else len(VAL_ROUTED)),
        "checkpoint_bundle_sha256_valid": canonical_sha256(components)
        == candidate_result["checkpoint_bundle_sha256"],
    }
    if not all(completion_assertions.values()):
        raise AssertionError(
            f"candidate completion hard gate 실패: {completion_assertions}"
        )
    completion_payload = {
        "schema_version": CANDIDATE_COMPLETION_SCHEMA,
        "status": "CANDIDATE_COMPLETE",
        "run_id": RUN_ID,
        "run_mode": RUN_MODE,
        "candidate_name": candidate_name,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
        "completed_epoch": EPOCHS,
        "best_epoch": best_epoch,
        "best_checkpoint": best_descriptor,
        "final_checkpoint": final_descriptor,
        "epoch_checkpoints": [
            checkpoint_descriptors[epoch]
            for epoch in range(1, EPOCHS + 1)
        ],
        "candidate_result": candidate_result,
        "assertions": completion_assertions,
    }
    completion_descriptor = upload_json_create_only(
        BUCKET,
        completion_payload,
        completion_object_name(candidate_name),
        CANDIDATE_COMPLETION_ARTIFACT,
        {
            "run_config_sha256": RUN_CONFIG_SHA256,
            "runtime_contract_sha256": RUNTIME_CONTRACT_SHA256,
            "candidate_name": candidate_name,
        },
    )
    candidate_results_by_name[candidate_name] = candidate_result
    candidate_completion_descriptors[
        candidate_name
    ] = completion_descriptor
    print(
        "CANDIDATE COMPLETE",
        candidate_name,
        completion_descriptor["uri"],
    )

    del component_models, optimizer, scheduler, scaler, best
    gc.collect()
    torch.cuda.empty_cache()

candidate_results = [
    candidate_results_by_name[candidate_name]
    for candidate_name in CANDIDATE_NAMES
]


CANDIDATE COMPLETE REUSED single_task gs://YOUR_GCS_BUCKET/dcc/mission3/v4/runs/m3-v4-yolo26s-fixed-router-obstacle-aux-v3/single_task/candidate_complete.json
CANDIDATE RESUMED multitask completed_epoch=2 next_epoch=3
multitask {'epoch': 3, 'full_frame_rows': 246308, 'full_frame_loss': 0.011165159279106548, 'roi_rows': 191640, 'roi_loss': 0.009850325099620413, 'val_overall_rmse_m': 0.21739925387652567, 'val_worst_type_rmse_m': 0.3720814135263699, 'val_worst_obstacle_rmse_m': 0.6819092067758541}
multitask {'epoch': 4, 'full_frame_rows': 246308, 'full_frame_loss': 0.009716695907947707, 'roi_rows': 191640, 'roi_loss': 0.007767191747403885, 'val_overall_rmse_m': 0.18626244054854849, 'val_worst_type_rmse_m': 0.3646121295973048, 'val_worst_obstacle_rmse_m': 0.3349668703209537}
multitask {'epoch': 5, 'full_frame_rows': 246308, 'full_frame_loss': 0.008635320250432963, 'roi_rows': 191640, 'roi_loss': 0.006223849928799349, 'val_overall_rmse_m': 0.18436134099658713, 'val_worst_type_rmse_m': 0.365

## 6. Candidate bundle and MODEL_DEPLOYMENT_FROZEN selection seal v2


In [ ]:
if {x["candidate_name"] for x in candidate_results} != set(CANDIDATE_NAMES):
    raise AssertionError("candidate completeness 오류")
if len({x["routing_sha256"] for x in candidate_results}) != 1:
    raise AssertionError("두 후보 routed input 불일치")
if len({canonical_sha256(x["components"]["yolo_detector"]) for x in candidate_results}) != 1:
    raise AssertionError("두 후보 detector 불일치")
validation_contracts = {
    (x["validation_rows"],x["validation_stem_sha256"],x["validation_target_sha256"],x["validation_obstruction_sha256"])
    for x in candidate_results
}
if len(validation_contracts) != 1:
    raise AssertionError("candidate validation contract 불일치")
leaderboard=sorted(candidate_results,key=lambda x:(
    x["metrics"]["overall_rmse_m"],x["metrics"]["worst_type_rmse_m"],
    x["metrics"]["worst_obstacle_rmse_m"],x["model_id"],
))
winner=leaderboard[0]
bundle={
    "status":"CANDIDATES_COMPLETE",
    "schema_version":"dcc-m3-v4-router-candidate-bundle-v2",
    "run_id":RUN_ID,"run_mode":RUN_MODE,
    "training_scope":"full" if RUN_MODE=="full" else "smoke",
    "eligible_for_selection":RUN_MODE=="full",
    "dataset_version":DATASET_VERSION,
    "dataset_seal_sha256":DATASET_SEAL_DESCRIPTOR["sha256"],
    "detector_config":DETECTOR_CONFIG,"detector_config_sha256":DETECTOR_CONFIG_SHA256,
    "detector_training_contract":DETECTOR_CONFIG,
    "routing_contract":ROUTING_CONTRACT,"routing_sha256":ROUTING_SHA256,
    "route_cache":{"train":TRAIN_ROUTE_CACHE_DESCRIPTOR,"val":VAL_ROUTE_CACHE_DESCRIPTOR},
    "candidates":candidate_results,
    "candidate_completion_markers":candidate_completion_descriptors,
    "preliminary_winner":winner["model_id"],
    "selection_rule":[
        "minimum validation overall meter RMSE",
        "minimum validation worst-type meter RMSE",
        "minimum validation worst-obstacle meter RMSE",
        "lexicographic model_id",
    ],
    "run_config":RUN_CONFIG,"run_config_sha256":RUN_CONFIG_SHA256,
}
bundle_descriptor=upload_json_create_only(
    BUCKET,bundle,f"{RUN_PREFIX}/candidate_bundle.json","m3_v4_router_candidate_bundle"
)
if RUN_MODE!="full":
    print("SMOKE COMPLETE",bundle_descriptor["uri"])
else:
    assertions={
        "candidate_count_exact":len(candidate_results)==2,
        "candidate_completion_markers_exact":set(candidate_completion_descriptors)==set(CANDIDATE_NAMES),
        "four_regressors_trained":sum(
            sum(k.endswith("_regressor") for k in x["components"]) for x in candidate_results
        )==4,
        "candidate_routed_inputs_identical":len({x["routing_sha256"] for x in candidate_results})==1,
        "shared_frozen_detector_between_arms":len({
            canonical_sha256(x["components"]["yolo_detector"]) for x in candidate_results
        })==1,
        "v4_detector_retrained_default":DETECTOR_MODE=="retrain_v4",
        "route_by_type_exact":winner["route_by_type"]==ROUTE_BY_TYPE,
        "validation_rows_exact":all(x["validation_rows"]==EXPECTED_VAL_N for x in candidate_results),
        "validation_route_counts_exact":VAL_ROUTE_COUNTS==EXPECTED_VAL_ROUTE_COUNTS,
        "checkpoint_bundle_sha256_valid":canonical_sha256(winner["components"])==winner["checkpoint_bundle_sha256"],
        "routing_sha256_valid":canonical_sha256(ROUTING_CONTRACT)==ROUTING_SHA256,
        "test_images_not_accessed":True,"private_truth_not_accessed":True,
        "dataset_assertions_passed":all(dataset_seal["assertions"].values()),
    }
    if not all(assertions.values()):
        raise AssertionError(f"selection hard gate 실패: {assertions}")
    selection_seal={
        "status":"MODEL_DEPLOYMENT_FROZEN",
        "schema_version":"dcc-m3-v4-router-model-selection-seal-v2",
        "selection_id":SELECTION_ID,
        "selection_mode":"fixed_test_validation_select",
        "training_scope":"full","dataset_version":DATASET_VERSION,
        "dataset_seal":DATASET_SEAL_DESCRIPTOR,
        "dataset_seal_sha256":DATASET_SEAL_DESCRIPTOR["sha256"],
        "dataset_config_sha256":dataset_seal["config_sha256"],
        "detector":DETECTOR_DESCRIPTOR,
        "detector_config":DETECTOR_CONFIG,
        "detector_config_sha256":DETECTOR_CONFIG_SHA256,
        "split_seal_sha256":dataset_seal["upstream"]["split_seal_sha256"],
        "candidate_bundle":bundle_descriptor,
        "candidate_bundle_sha256":bundle_descriptor["sha256"],
        "winner_model_id":winner["model_id"],"winner_kind":"type_router",
        "winner":winner,"route_by_type":ROUTE_BY_TYPE,
        "components":winner["components"],
        "checkpoint_bundle_sha256":winner["checkpoint_bundle_sha256"],
        "routing_contract":ROUTING_CONTRACT,"routing_sha256":ROUTING_SHA256,
        "detector_training_contract":DETECTOR_CONFIG,
        "inference_rule":INFERENCE_RULE,
        "leaderboard":[{
            "rank":rank,"model_id":x["model_id"],"candidate_name":x["candidate_name"],
            "kind":"type_router","overall_rmse_m":x["metrics"]["overall_rmse_m"],
            "worst_type_rmse_m":x["metrics"]["worst_type_rmse_m"],
            "worst_obstacle_rmse_m":x["metrics"]["worst_obstacle_rmse_m"],
        } for rank,x in enumerate(leaderboard,1)],
        "selection_rule":bundle["selection_rule"],
        "validation_contract":{
            "rows":EXPECTED_VAL_N,"stem_sha256":winner["validation_stem_sha256"],
            "target_sha256":winner["validation_target_sha256"],
            "obstruction_sha256":winner["validation_obstruction_sha256"],
            "route_counts":VAL_ROUTE_COUNTS,
        },
        "test_contract":TEST_CONTRACT,"obstacle_vocabulary":OBSTACLE_ORDER,
        "assertions":assertions,
    }
    selection_descriptor=upload_json_create_only(
        BUCKET,selection_seal,SELECTION_OBJECT,"m3_v4_router_model_selection_seal"
    )
    print("MODEL_DEPLOYMENT_FROZEN",winner["model_id"],selection_descriptor["uri"])


MODEL_DEPLOYMENT_FROZEN v4-fixed-router-multitask gs://YOUR_GCS_BUCKET/dcc/mission3/v4/selection/m3-selection-v4-router-fixedtest-v4/model_selection_seal.json
